# Overview

This comprehensive analysis provides:
- Original SysEngBench + all position variants (a, b, c, d) + OSQ format
- Position bias detection and quantification
- Format preference analysis (MCQ vs OSQ)
- Tokenomics & Cost Analysis
  - Token usage tracking for all models
  - Cost per sample calculation
  - ROI analysis for thinking vs standard models

**Key Insights Generated**
- Is the extra cost of thinking models justified?
- Which models show position bias (and how much does it cost)?
- At what scale do thinking models become cost-effective?
- Which format (MCQ vs OSQ) provides best value?

In [1]:
!pip install lm-eval datasets pandas numpy matplotlib seaborn scipy pyyaml tqdm -q


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Data Preparation

## Setup and Imports

 -*- coding: utf-8 -*-

SysEngBench Complete Analysis Notebook
Evaluates all SysEngBench variants and analyzes:
1. Answer position bias (A, B, C, D variants)
2. MCQ vs OSQ performance comparison
3. Cost modeling and Tokenomics Analysis


In [ ]:
# %%
import json
import os
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from datetime import datetime
import yaml
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# %%
# Configuration

MODELS = ["deepseek-r1:7b", "qwq:32b"]  # Add your models here
OUTPUT_BASE_DIR = Path("./sysengbench_results")

# SysEngBench variants
VARIANTS = {
    "sysengbench-a": "Answer in position A",
    "sysengbench-b": "Answer in position B", 
    "sysengbench-c": "Answer in position C",
    "sysengbench-d": "Answer in position D",
    "sysengbench-osq": "Open Short Question format"
}

## Data Loading, Load Results

## Taking inventory, file presence check

In [2]:
import json
from pathlib import Path
import pandas as pd


def scan_phase_outputs(phase_dirs, tasks_filter=None):
    """
    Scan multiple phase dirs (e.g. phase4, phase5) and return a DataFrame
    where each row = model and each col = task showing a compact status:
        "(results:✅ samples:✅ judged:❌)"
    """
    # --- DISCOVER ALL TASKS AND MODELS ---
    tasks = set()
    models = set()

    for base in phase_dirs:
        p = Path(base)
        if not p.exists():
            continue
        for task_dir in p.iterdir():
            if task_dir.is_dir():
                tasks.add(task_dir.name)
                for model_dir in task_dir.iterdir():
                    if model_dir.is_dir():
                        models.add(model_dir.name)

    if tasks_filter:
        tasks = {t for t in tasks if t in tasks_filter}

    tasks = sorted(tasks)
    models = sorted(models)

    df = pd.DataFrame(index=models, columns=tasks)

    # --- CHECK PRESENCE ---
    for model in models:
        for task in tasks:
            found_results = False
            found_samples = False
            found_judged = False

            for phase_root in phase_dirs:
                task_dir = Path(phase_root, task, model)
                if not task_dir.exists():
                    continue

                for f in task_dir.iterdir():
                    name = f.name

                    # PHASE-4 RESULTS
                    if name.startswith("results_") and f.suffix == ".json":
                        found_results = True

                    # PHASE-4 RAW SAMPLES
                    if name.startswith(f"samples_{task}_") and f.suffix == ".jsonl" and "__" not in name:
                        found_samples = True

                    # PHASE-5 JUDGED SAMPLES (heuristic V1)
                    if name.startswith(f"samples_{task}_") and "__" in name:
                        found_judged = True

            status = (
                f"(results:{'✅' if found_results else '❌'}  "
                f"samples:{'✅' if found_samples else '❌'}  "
                f"judged:{'✅' if found_judged else '❌'})"
            )
            df.at[model, task] = status

    return df


In [6]:
# phase4 = r"C:\Users\rabel\Desktop\dissertation\src\phase4_inference\downloaded_output"
phase4 = r"C:\Users\rabel\Desktop\dissertation-outputs\output"
phase5 = r"C:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge"

# df_status = scan_phase_outputs([phase4, phase5], tasks_filter=["sysengbench-osq"])
df_status = scan_phase_outputs([phase4, phase5])
# print(df_status)

# View in Jupyter
display(df_status)


,sysengbench,sysengbench-a,sysengbench-b,sysengbench-c,sysengbench-d,sysengbench-osq,unused-rubrics
deepseek-r1__14b,(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌ judged:❌)
deepseek-r1__32b,(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌ judged:❌)
deepseek-r1__8b,(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌ judged:❌)
devstral__24b,(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌ judged:❌)
gemma3__12b,(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌ judged:❌)
gemma3__1b,(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌ judged:❌)
gemma3__270m,(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌ judged:❌)
gemma3__27b,(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:✅),(results:❌ samples:❌ judged:❌)
gemma3__4b,(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:✅),(results:❌ samples:❌ judged:❌)
gemma3n__e2b,(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌ judged:❌)


In [9]:
import json
from pathlib import Path
import pandas as pd


def scan_phase_outputs(phase_dirs, tasks_filter=None, judged_tasks=None):
    """
    Scan multiple phase dirs (e.g. phase4, phase5) and return a DataFrame
    where each row = model and each col = task showing a compact status:
        "(results:✅ samples:✅ judged:❌)"
    """
    # --- DISCOVER ALL TASKS AND MODELS ---
    if judged_tasks is None:
        judged_tasks = set()   # nothing judged unless explicit
    
    tasks = set()
    models = set()

    for base in phase_dirs:
        p = Path(base)
        if not p.exists():
            continue
        for task_dir in p.iterdir():
            if task_dir.is_dir():
                tasks.add(task_dir.name)
                for model_dir in task_dir.iterdir():
                    if model_dir.is_dir():
                        models.add(model_dir.name)

    if tasks_filter:
        tasks = {t for t in tasks if t in tasks_filter}

    tasks = sorted(tasks)
    models = sorted(models)

    df = pd.DataFrame(index=models, columns=tasks)

    # --- CHECK PRESENCE ---
    for model in models:
        for task in tasks:
            found_results = False
            found_samples = False
            found_judged = False

            for phase_root in phase_dirs:
                task_dir = Path(phase_root, task, model)
                if not task_dir.exists():
                    continue

                for f in task_dir.iterdir():
                    name = f.name

                    if name.startswith("results_") and f.suffix == ".json":
                        found_results = True

                    if name.startswith(f"samples_{task}_") and f.suffix == ".jsonl" and "__" not in name:
                        found_samples = True

                    # only check judged if this task is allowed to have judged files
                    if task in judged_tasks:
                        if name.startswith(f"samples_{task}_") and "__" in name:
                            found_judged = True

            # 🔒 Override judged to ❌ if this task is not in judged_tasks
            if task not in judged_tasks:
                found_judged = False

            parts = [
                f"results:{'✅' if found_results else '❌'}",
                f"samples:{'✅' if found_samples else '❌'}",
            ]

            if task in judged_tasks:   # only include third field for those tasks
                parts.append(f"judged:{'✅' if found_judged else '❌'}")

            status = "(" + "  ".join(parts) + ")"

            df.at[model, task] = status

    return df

In [10]:
phase4 = r"C:\Users\rabel\Desktop\dissertation-outputs\output"
phase5 = r"C:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge"

df_status = scan_phase_outputs(
    [phase4, phase5],
    judged_tasks={"sysengbench-osq"}   # <–– only this task gets judged checks
)

display(df_status)


,sysengbench,sysengbench-a,sysengbench-b,sysengbench-c,sysengbench-d,sysengbench-osq,unused-rubrics
deepseek-r1__14b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌)
deepseek-r1__32b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌)
deepseek-r1__8b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌)
devstral__24b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌)
gemma3__12b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌)
gemma3__1b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌)
gemma3__270m,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌)
gemma3__27b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:✅),(results:❌ samples:❌)
gemma3__4b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:✅),(results:❌ samples:❌)
gemma3n__e2b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌),(results:❌ samples:❌)


removing the `unused-rubrics` column from discovery

In [12]:
import json
from pathlib import Path
import pandas as pd


def scan_phase_outputs(phase_dirs, tasks_filter=None, judged_tasks=None):
    if judged_tasks is None:
        judged_tasks = set()

    # -------- helper: does model_dir contain any results/samples files? --------
    def model_has_json_files(model_dir: Path) -> bool:
        for f in model_dir.iterdir():
            if not f.is_file():
                continue
            if f.suffix in {".json", ".jsonl"} and (
                f.name.startswith("results_") or f.name.startswith("samples_")
            ):
                return True
        return False

    # -------- helper: is this folder actually a "task"? --------
    # Only if it contains at least one model subfolder that has relevant files
    def folder_is_task(task_dir: Path) -> bool:
        for m in task_dir.iterdir():
            if m.is_dir() and model_has_json_files(m):
                return True
        return False

    # -------- discover tasks and models --------
    tasks = set()
    models = set()

    for base in phase_dirs:
        p = Path(base)
        if not p.exists():
            continue
        for task_dir in p.iterdir():
            if not task_dir.is_dir():
                continue
            if not folder_is_task(task_dir):
                continue
            task = task_dir.name
            tasks.add(task)
            for model_dir in task_dir.iterdir():
                if model_dir.is_dir() and model_has_json_files(model_dir):
                    models.add(model_dir.name)

    # apply optional task filtering
    if tasks_filter:
        tasks = {t for t in tasks if t in tasks_filter}

    tasks = sorted(tasks)
    models = sorted(models)

    # -------- build dataframe --------
    df = pd.DataFrame(index=models, columns=tasks)

    for model in models:
        for task in tasks:
            found_results = False
            found_samples = False
            found_judged = False

            for base in phase_dirs:
                task_dir = Path(base, task, model)
                if not task_dir.exists():
                    continue
                for f in task_dir.iterdir():
                    name = f.name
                    # phase-4 results
                    if name.startswith("results_") and f.suffix == ".json":
                        found_results = True
                    # phase-4 raw samples
                    if name.startswith(f"samples_{task}_") and f.suffix == ".jsonl" and "__" not in name:
                        found_samples = True
                    # phase-5 judged samples
                    if task in judged_tasks:
                        if name.startswith(f"samples_{task}_") and "__" in name:
                            found_judged = True

            # if task is not judged-eligible, do not display judged at all
            parts = [
                f"results:{'✅' if found_results else '❌'}",
                f"samples:{'✅' if found_samples else '❌'}",
            ]
            if task in judged_tasks:
                parts.append(f"judged:{'✅' if found_judged else '❌'}")

            df.at[model, task] = "(" + "  ".join(parts) + ")"

    return df


In [ ]:
# --- Define your phase directories ---
phase4 = r"C:\Users\rabel\Desktop\dissertation-outputs\output"
phase5 = r"C:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge"

# --- Call the function ---
df_status = scan_phase_outputs(
    phase_dirs=[phase4, phase5],
    # tasks_filter=["sysengbench-osq"],
    judged_tasks={"sysengbench-osq"}   # only this task shows judged:✅/❌
)

# --- View in Jupyter ---
display(df_status)


,sysengbench,sysengbench-a,sysengbench-b,sysengbench-c,sysengbench-d,sysengbench-osq
deepseek-r1__14b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌)
deepseek-r1__32b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌)
deepseek-r1__8b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌)
devstral__24b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌)
gemma3__12b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌)
gemma3__1b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌)
gemma3__270m,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌)
gemma3__27b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:✅)
gemma3__4b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:✅)
gemma3n__e2b,(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅),(results:✅ samples:✅ judged:❌)


## Now let's calculate the results for the judged tasks (sysengbench-osq)

This allows us to then compare apples to apples with statistics etc

### Version 0

Same code from runpod cli multiple workers to check what's been done

In [ ]:
base_output_dir = r"C:\Users\rabel\Desktop\dissertation-outputs\output"

In [ ]:
import json
import pandas as pd
from pathlib import Path

def generate_model_task_matrix_with_scores_and_samples(base_output_dir: str) -> pd.DataFrame:
    """
    Build a matrix with:
      * 'Model Folder Name' (safe for filesystem)
      * 'Model Ollama Name' (with ':' restored)
      * One column per task showing ✅/❌ with counts, best score, and sample count.

    Directory structure expected:
        base_output_dir/task_name/model_folder_name/
            results_<timestamp>.json
            samples_<task>_<timestamp>.jsonl
    """
    base_path = Path(base_output_dir)
    if not base_path.exists():
        raise FileNotFoundError(f"Directory does not exist: {base_output_dir}")

    # Discover all tasks and all folder names
    task_names = [p.name for p in base_path.iterdir() if p.is_dir()]
    model_folder_names = set()
    for task in task_names:
        for model_dir in (base_path / task).iterdir():
            if model_dir.is_dir():
                model_folder_names.add(model_dir.name)

    # Prepare matrix with two leading columns
    matrix = pd.DataFrame(index=sorted(model_folder_names),
                          columns=["Model Folder Name", "Model Ollama Name"] + sorted(task_names))

    for model_folder in sorted(model_folder_names):
        # Fill in the two identifying columns
        matrix.at[model_folder, "Model Folder Name"] = model_folder
        # Convert "__" back to ":" for correct Ollama name
        matrix.at[model_folder, "Model Ollama Name"] = model_folder.replace("__", ":")

        for task in task_names:
            model_dir = base_path / task / model_folder
            if not model_dir.exists():
                matrix.at[model_folder, task] = "❌ (0 – max:0.0 – samples:0)"
                continue

            result_files = [f for f in model_dir.iterdir()
                            if f.name.startswith("results_") and f.suffix == ".json"]

            if not result_files:
                matrix.at[model_folder, task] = "❌ (0 – max:0.0 – samples:0)"
                continue

            best_score = 0.0
            best_result_file = None

            # Find highest scoring results file
            for rf in result_files:
                try:
                    with open(rf, "r") as fh:
                        data = json.load(fh)
                    if "results" in data and task in data["results"]:
                        task_data = data["results"][task]
                        score = (task_data.get("exact_match,strict-match")
                                 or task_data.get("exact_match")
                                 or max((v for v in task_data.values() if isinstance(v,(int,float))), default=0.0))
                        if score and score > best_score:
                            best_score = float(score)
                            best_result_file = rf
                except Exception as e:
                    print(f"Warning: could not parse {rf}: {e}")

            n_results = len(result_files)
            samples_exist = False
            max_samples_count = 0

            if best_result_file:
                ts = best_result_file.stem.replace("results_", "")
                expected_samples_prefix = f"samples_{task}_{ts}"
                for f in model_dir.iterdir():
                    if f.name.startswith(expected_samples_prefix) and f.suffix == ".jsonl":
                        samples_exist = True
                        with open(f, "r", encoding="utf-8") as sf:
                            count = sum(1 for _ in sf)
                        max_samples_count = count
                        break

            if best_result_file and samples_exist:
                matrix.at[model_folder, task] = f"✅ ({n_results}) – max:{best_score:.3f} – samples:{max_samples_count}"
            else:
                matrix.at[model_folder, task] = f"❌ ({n_results}) – max:{best_score:.3f} – samples:{max_samples_count}"

    matrix.index.name = "Model Folder Name (index)"
    return matrix


In [3]:
matrix_df = generate_model_task_matrix_with_scores_and_samples(base_output_dir)

matrix_df = matrix_df.reset_index(drop=True)

pd.set_option('display.expand_frame_repr', False)  # don’t wrap whole rows
pd.set_option('display.max_colwidth', None)        # don’t truncate/wrap cells

# View in Jupyter
display(matrix_df)

# Optional export
# matrix_df.to_csv("model_task_matrix_with_scores_and_samples.csv")


,Model Folder Name,Model Ollama Name,sysengbench,sysengbench-a,sysengbench-b,sysengbench-c,sysengbench-d,sysengbench-osq-synthetic
0,deepseek-r1__14b,deepseek-r1:14b,✅ (1) – max:0.142 – samples:1144,✅ (2) – max:0.647 – samples:1144,✅ (2) – max:0.092 – samples:1144,✅ (2) – max:0.085 – samples:1144,✅ (2) – max:0.180 – samples:1144,❌ (0 – max:0.0 – samples:0)
1,deepseek-r1__32b,deepseek-r1:32b,✅ (1) – max:0.145 – samples:1144,✅ (2) – max:0.656 – samples:1144,✅ (2) – max:0.087 – samples:1144,✅ (2) – max:0.079 – samples:1144,✅ (2) – max:0.176 – samples:1144,❌ (0 – max:0.0 – samples:0)
2,deepseek-r1__8b,deepseek-r1:8b,❌ (0 – max:0.0 – samples:0),❌ (1) – max:0.000 – samples:0,❌ (1) – max:0.000 – samples:0,❌ (1) – max:0.000 – samples:0,❌ (1) – max:0.000 – samples:0,❌ (0 – max:0.0 – samples:0)
3,devstral__24b,devstral:24b,✅ (1) – max:0.931 – samples:1144,✅ (1) – max:0.898 – samples:1144,✅ (1) – max:0.941 – samples:1144,✅ (1) – max:0.941 – samples:1144,✅ (1) – max:0.911 – samples:1144,❌ (0 – max:0.0 – samples:0)
4,gemma3__12b,gemma3:12b,✅ (2) – max:0.892 – samples:1144,✅ (2) – max:0.865 – samples:1144,✅ (2) – max:0.894 – samples:1144,✅ (2) – max:0.906 – samples:1144,✅ (1) – max:0.892 – samples:1144,❌ (0 – max:0.0 – samples:0)
5,gemma3__1b,gemma3:1b,✅ (2) – max:0.691 – samples:1144,✅ (2) – max:0.748 – samples:1144,✅ (2) – max:0.552 – samples:1144,✅ (2) – max:0.794 – samples:1144,✅ (1) – max:0.753 – samples:1144,❌ (0 – max:0.0 – samples:0)
6,gemma3__270m,gemma3:270m,✅ (2) – max:0.136 – samples:1144,✅ (2) – max:0.108 – samples:1144,✅ (2) – max:0.007 – samples:1144,✅ (2) – max:0.317 – samples:1144,✅ (1) – max:0.010 – samples:1144,❌ (0 – max:0.0 – samples:0)
7,gemma3__27b,gemma3:27b,✅ (2) – max:0.920 – samples:1144,✅ (2) – max:0.903 – samples:1144,✅ (2) – max:0.918 – samples:1144,✅ (2) – max:0.932 – samples:1144,✅ (1) – max:0.911 – samples:1144,❌ (0 – max:0.0 – samples:0)
8,gemma3__4b,gemma3:4b,✅ (2) – max:0.863 – samples:1144,✅ (2) – max:0.828 – samples:1144,✅ (2) – max:0.844 – samples:1144,✅ (2) – max:0.876 – samples:1144,✅ (1) – max:0.847 – samples:1144,❌ (0 – max:0.0 – samples:0)
9,gemma3n__e2b,gemma3n:e2b,✅ (2) – max:0.865 – samples:1144,✅ (2) – max:0.816 – samples:1144,✅ (2) – max:0.898 – samples:1144,✅ (2) – max:0.876 – samples:1144,✅ (1) – max:0.792 – samples:1144,❌ (0 – max:0.0 – samples:0)


### Version 1

In [ ]:
class SysEngBenchAnalyzer:
    """Analyzer for SysEngBench evaluation results"""
    
    def __init__(self, results_dir: Path):
        self.results_dir = Path(results_dir)
        self.all_results = {}
        self.load_all_results()
        
    def load_all_results(self):
        """Load all results from output directories"""
        for result_dir in self.results_dir.glob("*"):
            if result_dir.is_dir():
                # Parse model and variant from directory name
                parts = result_dir.name.split("_")
                if len(parts) >= 2:
                    model = "_".join(parts[:-1])
                    variant = parts[-1]
                    
                    # Load results
                    results_file = result_dir / "results.json"
                    samples_file = list(result_dir.glob("**/samples*.jsonl"))
                    
                    if results_file.exists():
                        with open(results_file, 'r') as f:
                            results = json.load(f)
                        
                        samples = []
                        if samples_file:
                            with open(samples_file[0], 'r') as f:
                                for line in f:
                                    samples.append(json.loads(line))
                        
                        if model not in self.all_results:
                            self.all_results[model] = {}
                        
                        self.all_results[model][variant] = {
                            "results": results,
                            "samples": samples
                        }
    
    def get_accuracy(self, model: str, variant: str) -> float:
        """Extract accuracy from results"""
        try:
            results = self.all_results[model][variant]["results"]
            # Navigate through the results structure
            for task_name, task_results in results.get("results", {}).items():
                if "exact_match" in task_results:
                    return task_results["exact_match,none"]
                elif "acc" in task_results:
                    return task_results["acc,none"]
            return 0.0
        except:
            return 0.0
    
    def analyze_position_bias(self) -> pd.DataFrame:
        """Analyze answer position bias across MCQ variants"""
        data = []
        
        for model in self.all_results.keys():
            mcq_variants = ["sysengbench-a", "sysengbench-b", 
                          "sysengbench-c", "sysengbench-d"]
            
            accuracies = {}
            for variant in mcq_variants:
                if variant in self.all_results[model]:
                    accuracies[variant[-1].upper()] = self.get_accuracy(model, variant)
            
            if accuracies:
                data.append({
                    "Model": model,
                    "Position A": accuracies.get("A", 0),
                    "Position B": accuracies.get("B", 0),
                    "Position C": accuracies.get("C", 0),
                    "Position D": accuracies.get("D", 0),
                    "Mean": np.mean(list(accuracies.values())),
                    "Std Dev": np.std(list(accuracies.values())),
                    "Max-Min": max(accuracies.values()) - min(accuracies.values())
                })
        
        return pd.DataFrame(data)
    
    def analyze_mcq_vs_osq(self) -> pd.DataFrame:
        """Compare MCQ vs OSQ performance"""
        data = []
        
        for model in self.all_results.keys():
            # Get MCQ average (across all positions)
            mcq_accs = []
            for variant in ["sysengbench-a", "sysengbench-b", 
                          "sysengbench-c", "sysengbench-d"]:
                if variant in self.all_results[model]:
                    mcq_accs.append(self.get_accuracy(model, variant))
            
            mcq_avg = np.mean(mcq_accs) if mcq_accs else 0
            
            # Get OSQ accuracy
            osq_acc = 0
            if "sysengbench-osq" in self.all_results[model]:
                osq_acc = self.get_accuracy(model, "sysengbench-osq")
            
            data.append({
                "Model": model,
                "MCQ Average": mcq_avg,
                "OSQ": osq_acc,
                "Difference (MCQ-OSQ)": mcq_avg - osq_acc,
                "Format Preference": "MCQ" if mcq_avg > osq_acc else "OSQ"
            })
        
        return pd.DataFrame(data)
    
    def get_question_level_analysis(self, model: str) -> pd.DataFrame:
        """Analyze performance at question level across variants"""
        question_data = []
        
        for variant in self.all_results[model].keys():
            samples = self.all_results[model][variant]["samples"]
            
            for sample in samples:
                doc = sample.get("doc", {})
                target = sample.get("target", "")
                pred = sample.get("pred", "")
                
                question_data.append({
                    "Model": model,
                    "Variant": variant,
                    "Question": doc.get("Question", ""),
                    "Category": doc.get("Category", ""),
                    "SubCategory": doc.get("Sub-Category", ""),
                    "Target": target,
                    "Prediction": pred,
                    "Correct": target == pred,
                    "Answer_Position": variant[-1].upper() if variant != "sysengbench-osq" else "OSQ"
                })
        
        return pd.DataFrame(question_data)

In [ ]:
# Initialize analyzer
analyzer = SysEngBenchAnalyzer(OUTPUT_BASE_DIR)

## Filter / Adjust for thinking models

Need to address the `<think>` mechanism for thinking models. 

In [ ]:
thinking_models = ["deepseek-r1:7b", "qwq:32b"]  # Add your thinking models here

In [ ]:
# Filter / Adjust for thinking models
# Need to address the `<think>` mechanism for thinking models.

# Simple Data Analysis

## Answer Position Bias Analysis

In [ ]:

# Analyze position bias
position_bias_df = analyzer.analyze_position_bias()
print("📊 Answer Position Bias Analysis:")
print(position_bias_df.to_string(index=False))

# %%
# Visualize position bias
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Bar chart of accuracy by position for each model
ax = axes[0, 0]
positions = ['Position A', 'Position B', 'Position C', 'Position D']
x = np.arange(len(positions))
width = 0.35

for i, (_, row) in enumerate(position_bias_df.iterrows()):
    offset = width * i
    ax.bar(x + offset, [row[pos] for pos in positions], 
           width, label=row['Model'])

ax.set_xlabel('Answer Position')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Answer Position')
ax.set_xticks(x + width / 2)
ax.set_xticklabels(['A', 'B', 'C', 'D'])
ax.legend()
ax.set_ylim(0, 1)

# 2. Heatmap of position bias
ax = axes[0, 1]
heatmap_data = position_bias_df[positions].T
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='coolwarm', 
            center=heatmap_data.mean().mean(), ax=ax)
ax.set_title('Position Bias Heatmap')
ax.set_xlabel('Model')
ax.set_ylabel('Position')

# 3. Variance analysis
ax = axes[1, 0]
ax.bar(position_bias_df['Model'], position_bias_df['Std Dev'])
ax.set_xlabel('Model')
ax.set_ylabel('Standard Deviation')
ax.set_title('Variance in Performance Across Positions')
ax.tick_params(axis='x', rotation=45)

# 4. Max-Min spread
ax = axes[1, 1]
ax.bar(position_bias_df['Model'], position_bias_df['Max-Min'])
ax.set_xlabel('Model')
ax.set_ylabel('Max - Min Accuracy')
ax.set_title('Range of Performance Across Positions')
ax.tick_params(axis='x', rotation=45)

# Add a horizontal line for reference
ax.axhline(y=0.1, color='r', linestyle='--', alpha=0.5, label='10% threshold')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_BASE_DIR / 'position_bias_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## MCQ vs OSQ

In [ ]:
# %% [markdown]
# ## 6. MCQ vs OSQ Performance Comparison

# %%
# Analyze MCQ vs OSQ
mcq_osq_df = analyzer.analyze_mcq_vs_osq()
print("\n📝 MCQ vs OSQ Performance:")
print(mcq_osq_df.to_string(index=False))

# %%
# Visualize MCQ vs OSQ comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Side-by-side comparison
ax = axes[0]
x = np.arange(len(mcq_osq_df))
width = 0.35

ax.bar(x - width/2, mcq_osq_df['MCQ Average'], width, label='MCQ', alpha=0.8)
ax.bar(x + width/2, mcq_osq_df['OSQ'], width, label='OSQ', alpha=0.8)

ax.set_xlabel('Model')
ax.set_ylabel('Accuracy')
ax.set_title('MCQ vs OSQ Performance')
ax.set_xticks(x)
ax.set_xticklabels(mcq_osq_df['Model'], rotation=45)
ax.legend()
ax.set_ylim(0, 1)

# 2. Difference plot
ax = axes[1]
colors = ['green' if x > 0 else 'red' for x in mcq_osq_df['Difference (MCQ-OSQ)']]
ax.bar(mcq_osq_df['Model'], mcq_osq_df['Difference (MCQ-OSQ)'], color=colors)
ax.set_xlabel('Model')
ax.set_ylabel('MCQ - OSQ Accuracy')
ax.set_title('Performance Difference (MCQ - OSQ)')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.tick_params(axis='x', rotation=45)

# 3. Scatter plot
ax = axes[2]
ax.scatter(mcq_osq_df['MCQ Average'], mcq_osq_df['OSQ'], s=100)
for i, model in enumerate(mcq_osq_df['Model']):
    ax.annotate(model, (mcq_osq_df['MCQ Average'].iloc[i], mcq_osq_df['OSQ'].iloc[i]),
                xytext=(5, 5), textcoords='offset points', fontsize=8)

# Add diagonal line (equal performance)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('MCQ Average Accuracy')
ax.set_ylabel('OSQ Accuracy')
ax.set_title('MCQ vs OSQ Correlation')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(OUTPUT_BASE_DIR / 'mcq_vs_osq_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Question-Level Consistency Analysis -- Not sure what I want to do with this... 

In [ ]:
# Analyze consistency across variants for the same questions
def analyze_question_consistency(analyzer, model):
    """Analyze how consistently a model answers the same question across variants"""
    
    df = analyzer.get_question_level_analysis(model)
    
    # Group by question and calculate consistency
    consistency_data = []
    
    for question in df['Question'].unique():
        q_df = df[df['Question'] == question]
        
        # MCQ consistency (across a, b, c, d)
        mcq_df = q_df[q_df['Answer_Position'] != 'OSQ']
        mcq_consistency = mcq_df['Correct'].mean() if len(mcq_df) > 0 else np.nan
        
        # Overall consistency
        overall_consistency = q_df['Correct'].mean()
        
        # Check if all MCQ answers are the same
        mcq_predictions = mcq_df['Prediction'].values
        mcq_unanimous = len(set(mcq_predictions)) == 1 if len(mcq_predictions) > 1 else False
        
        consistency_data.append({
            'Question': question[:50] + '...' if len(question) > 50 else question,
            'Category': q_df.iloc[0]['Category'],
            'MCQ_Consistency': mcq_consistency,
            'Overall_Consistency': overall_consistency,
            'MCQ_Unanimous': mcq_unanimous,
            'Num_Variants_Tested': len(q_df)
        })
    
    return pd.DataFrame(consistency_data)

# Analyze for each model
for model in analyzer.all_results.keys():
    print(f"\n🔍 Question Consistency Analysis for {model}:")
    consistency_df = analyze_question_consistency(analyzer, model)
    
    # Summary statistics
    print(f"  - Questions always correct across MCQ variants: {(consistency_df['MCQ_Consistency'] == 1.0).sum()}")
    print(f"  - Questions always wrong across MCQ variants: {(consistency_df['MCQ_Consistency'] == 0.0).sum()}")
    print(f"  - Questions with mixed results: {((consistency_df['MCQ_Consistency'] > 0) & (consistency_df['MCQ_Consistency'] < 1)).sum()}")
    print(f"  - Average MCQ consistency: {consistency_df['MCQ_Consistency'].mean():.3f}")
    print(f"  - MCQ unanimous predictions: {consistency_df['MCQ_Unanimous'].sum()}/{len(consistency_df)}")

📈 Output Files Generated:

position_bias_analysis.png - Visual comparison of position preferences
mcq_vs_osq_analysis.png - Format comparison charts
position_bias_analysis.csv - Detailed position bias data
mcq_vs_osq_analysis.csv - Format comparison data
{model}_detailed.csv - Question-level results for each model

# Tokenomics analysis with regular vs thinking models

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
SysEngBench Complete Analysis with Tokenomics
Evaluates all SysEngBench variants including original, analyzes position bias,
and performs cost-benefit analysis of thinking models vs standard models
"""

# %% [markdown]
# # SysEngBench Complete Analysis with Tokenomics & Cost-Benefit
# 
# This notebook:
# 1. Runs original SysEngBench + all variants (a, b, c, d, osq)
# 2. Analyzes answer position bias and format preferences
# 3. **NEW: Tokenomics analysis - cost vs accuracy tradeoff**
# 4. **NEW: ROI calculation for thinking models**
# 5. Generates comprehensive reports with cost-benefit insights

# %% [markdown]
# ## 1. Setup and Configuration

# %%
import json
import os
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
from datetime import datetime
import yaml
from tqdm import tqdm
import warnings
from dataclasses import dataclass
import time
import re
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# %%
# Configuration
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# Models with their configurations and costs
@dataclass
class ModelConfig:
    name: str
    display_name: str
    is_thinking_model: bool
    max_tokens: int
    cost_per_1k_input: float  # USD per 1K tokens
    cost_per_1k_output: float  # USD per 1K tokens
    
MODELS_CONFIG = [
    # Thinking models
    ModelConfig("deepseek-r1:7b", "DeepSeek-R1 7B", True, 10000, 0.0001, 0.0002),
    ModelConfig("qwq:32b", "QwQ 32B", True, 12000, 0.0003, 0.0006),
    
    # Standard models for comparison
    ModelConfig("llama3.2:3b", "Llama 3.2 3B", False, 2048, 0.00005, 0.0001),
    ModelConfig("mistral:7b", "Mistral 7B", False, 2048, 0.00008, 0.00015),
]

OUTPUT_BASE_DIR = Path("./sysengbench_tokenomics")
OUTPUT_BASE_DIR.mkdir(exist_ok=True)

# All SysEngBench variants
VARIANTS = {
    "sysengbench": "Original SysEngBench (mixed positions)",
    "sysengbench-a": "Answer always in position A",
    "sysengbench-b": "Answer always in position B", 
    "sysengbench-c": "Answer always in position C",
    "sysengbench-d": "Answer always in position D",
    "sysengbench-osq": "Open Short Question format"
}

# %% [markdown]
# ## 2. Enhanced Task Configuration with Token Tracking

# %%
def create_task_config_with_tokenomics(variant: str, model_config: ModelConfig, output_dir: Path) -> Path:
    """
    Create task configuration with token tracking for cost analysis
    """
    task_dir = output_dir / "tasks"
    task_dir.mkdir(exist_ok=True)
    
    # Base configuration
    base_config = {
        "group": "sysengbench",
        "task": f"sysengbench_{variant}_{model_config.name.replace(':', '_')}",
        "dataset_path": f"rabell/{variant}",
        "dataset_name": "default",
        "test_split": "test",
        "num_fewshot": 0,
    }
    
    # Adjust generation kwargs based on model type
    if model_config.is_thinking_model:
        gen_kwargs = {
            "until": [],  # No early stopping for thinking models
            "max_gen_toks": model_config.max_tokens,
            "temperature": 0.0,
            "do_sample": False
        }
    else:
        gen_kwargs = {
            "until": ["\n\n"],  # Standard stopping for regular models
            "max_gen_toks": min(model_config.max_tokens, 512),  # Shorter for standard models
            "temperature": 0.0,
            "do_sample": False
        }
    
    if variant == "sysengbench-osq":
        # Open Short Question configuration
        config = {
            **base_config,
            "output_type": "generate_until",
            "generation_kwargs": gen_kwargs,
            "doc_to_text": """Category: {{Category}}
Sub-Category: {{Sub-Category}}

Question: {{Question}}

Provide a concise answer to this systems engineering question:""",
            "doc_to_target": "{{Answer}}",
            "metric_list": [
                {"metric": "exact_match", "aggregation": "mean", "higher_is_better": True},
                {"metric": "bleu", "aggregation": "mean", "higher_is_better": True}
            ]
        }
    else:
        # MCQ configuration
        prompt = """Category: {{Category}}
Sub-Category: {{Sub-Category}}

Question: {{Question}}

Options:
A. {{Choice A}}
B. {{Choice B}}
C. {{Choice C}}
D. {{Choice D}}

"""
        if model_config.is_thinking_model:
            prompt += "Analyze this systems engineering question step-by-step and provide your answer as a single letter (A, B, C, or D):"
        else:
            prompt += "Select the correct answer (A, B, C, or D):"
        
        config = {
            **base_config,
            "output_type": "generate_until",
            "generation_kwargs": gen_kwargs,
            "doc_to_text": prompt,
            "doc_to_target": "{{Answer}}",
            "filter_list": [
                {
                    "name": "extract_answer",
                    "filter": [
                        {
                            "function": "regex",
                            "regex_pattern": r"(?i)(?:final answer|answer is|therefore|select|choose)[:\s]*([A-D])",
                            "group": 1
                        },
                        {"function": "take_first"}
                    ]
                }
            ],
            "metric_list": [
                {"metric": "exact_match", "aggregation": "mean", "higher_is_better": True}
            ]
        }
    
    # Save configuration
    task_file = task_dir / f"{variant}_{model_config.name.replace(':', '_')}.yaml"
    with open(task_file, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    
    return task_file

# %% [markdown]
# ## 3. Enhanced Evaluation with Token Counting

# %%
@dataclass
class EvaluationResult:
    model: str
    variant: str
    accuracy: float
    total_samples: int
    avg_input_tokens: int
    avg_output_tokens: int
    total_input_tokens: int
    total_output_tokens: int
    avg_response_time: float
    total_cost: float
    cost_per_sample: float
    accuracy_per_dollar: float

def count_tokens_approximate(text: str) -> int:
    """Approximate token count (rough estimate: 1 token ≈ 4 chars or 0.75 words)"""
    return max(len(text) // 4, len(text.split()) * 4 // 3)

def run_evaluation_with_tokenomics(model_config: ModelConfig, variant: str, limit: Optional[int] = None) -> EvaluationResult:
    """
    Run evaluation with token tracking for cost analysis
    """
    print(f"🚀 Evaluating {model_config.display_name} on {variant}...")
    start_time = time.time()
    
    # Create task configuration
    task_file = create_task_config_with_tokenomics(variant, model_config, OUTPUT_BASE_DIR)
    
    # Output directory for this run
    output_dir = OUTPUT_BASE_DIR / f"{model_config.name.replace(':', '_')}_{variant}"
    
    # Build lm_eval command
    cmd = [
        "lm_eval",
        "--model", "local-chat-completions",
        "--model_args", f"model={model_config.name},base_url={OLLAMA_BASE_URL},max_tokens={model_config.max_tokens}",
        "--tasks", f"sysengbench_{variant}_{model_config.name.replace(':', '_')}",
        "--include_path", str(task_file.parent),
        "--output_path", str(output_dir),
        "--batch_size", "1",
        "--log_samples",
        "--verbosity", "WARNING"
    ]
    
    if model_config.is_thinking_model:
        cmd.extend(["--gen_kwargs", json.dumps({"until": [], "max_gen_toks": model_config.max_tokens})])
    
    if limit:
        cmd.extend(["--limit", str(limit)])
    
    # Run evaluation
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True, timeout=7200)
        
        # Load results
        results_file = output_dir / "results.json"
        samples_file = list(output_dir.glob("**/samples*.jsonl"))
        
        if results_file.exists():
            with open(results_file, 'r') as f:
                results = json.load(f)
            
            # Extract accuracy
            accuracy = 0.0
            for task_name, task_results in results.get("results", {}).items():
                if "exact_match,none" in task_results:
                    accuracy = task_results["exact_match,none"]
                elif "acc,none" in task_results:
                    accuracy = task_results["acc,none"]
                break
            
            # Analyze tokens from samples
            total_input_tokens = 0
            total_output_tokens = 0
            total_samples = 0
            
            if samples_file:
                with open(samples_file[0], 'r') as f:
                    for line in f:
                        sample = json.loads(line)
                        
                        # Count input tokens (prompt)
                        if 'doc' in sample:
                            prompt_text = json.dumps(sample['doc'])
                            total_input_tokens += count_tokens_approximate(prompt_text)
                        
                        # Count output tokens (response)
                        if 'resps' in sample and sample['resps']:
                            response = sample['resps'][0][0] if isinstance(sample['resps'][0], list) else sample['resps'][0]
                            total_output_tokens += count_tokens_approximate(str(response))
                        
                        total_samples += 1
            
            # Calculate costs
            input_cost = (total_input_tokens / 1000) * model_config.cost_per_1k_input
            output_cost = (total_output_tokens / 1000) * model_config.cost_per_1k_output
            total_cost = input_cost + output_cost
            
            # Calculate metrics
            avg_input = total_input_tokens / max(total_samples, 1)
            avg_output = total_output_tokens / max(total_samples, 1)
            cost_per_sample = total_cost / max(total_samples, 1)
            accuracy_per_dollar = accuracy / max(total_cost, 0.0001)  # Avoid division by zero
            
            elapsed_time = time.time() - start_time
            
            return EvaluationResult(
                model=model_config.name,
                variant=variant,
                accuracy=accuracy,
                total_samples=total_samples,
                avg_input_tokens=avg_input,
                avg_output_tokens=avg_output,
                total_input_tokens=total_input_tokens,
                total_output_tokens=total_output_tokens,
                avg_response_time=elapsed_time / max(total_samples, 1),
                total_cost=total_cost,
                cost_per_sample=cost_per_sample,
                accuracy_per_dollar=accuracy_per_dollar
            )
            
    except Exception as e:
        print(f"❌ Error evaluating {model_config.name} on {variant}: {e}")
        return EvaluationResult(
            model=model_config.name, variant=variant, accuracy=0, total_samples=0,
            avg_input_tokens=0, avg_output_tokens=0, total_input_tokens=0,
            total_output_tokens=0, avg_response_time=0, total_cost=0,
            cost_per_sample=0, accuracy_per_dollar=0
        )

# %% [markdown]
# ## 4. Run Complete Evaluation Suite

# %%
# Run evaluations with tokenomics tracking
LIMIT = 50  # Set to None for full evaluation

all_results = {}
tokenomics_data = []

for model_config in MODELS_CONFIG:
    model_results = {}
    
    for variant in VARIANTS.keys():
        result = run_evaluation_with_tokenomics(model_config, variant, limit=LIMIT)
        model_results[variant] = result
        tokenomics_data.append(result)
        
        print(f"✅ {model_config.display_name} on {variant}:")
        print(f"   Accuracy: {result.accuracy:.2%}")
        print(f"   Tokens: {result.avg_output_tokens:.0f} output, {result.avg_input_tokens:.0f} input")
        print(f"   Cost per sample: ${result.cost_per_sample:.6f}")
        print(f"   Accuracy per dollar: {result.accuracy_per_dollar:.1f}")
    
    all_results[model_config.name] = model_results

# Convert to DataFrame for analysis
tokenomics_df = pd.DataFrame([
    {
        'Model': r.model,
        'Variant': r.variant,
        'Accuracy': r.accuracy,
        'Avg Output Tokens': r.avg_output_tokens,
        'Avg Input Tokens': r.avg_input_tokens,
        'Total Cost': r.total_cost,
        'Cost per Sample': r.cost_per_sample,
        'Accuracy per Dollar': r.accuracy_per_dollar,
        'Response Time': r.avg_response_time
    }
    for r in tokenomics_data
])

# %% [markdown]
# ## 5. Tokenomics Analysis - Cost vs Accuracy Tradeoff

# %%
# Separate thinking models from standard models
thinking_models = [m.name for m in MODELS_CONFIG if m.is_thinking_model]
standard_models = [m.name for m in MODELS_CONFIG if not m.is_thinking_model]

# Calculate aggregate metrics
summary_df = tokenomics_df.groupby('Model').agg({
    'Accuracy': 'mean',
    'Avg Output Tokens': 'mean',
    'Total Cost': 'sum',
    'Cost per Sample': 'mean',
    'Accuracy per Dollar': 'mean'
}).round(4)

print("📊 TOKENOMICS SUMMARY")
print("="*60)
print(summary_df.to_string())

# %% [markdown]
# ## 6. Visualize Cost-Benefit Analysis

# %%
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Accuracy vs Cost Scatter
ax = axes[0, 0]
for model in tokenomics_df['Model'].unique():
    model_data = tokenomics_df[tokenomics_df['Model'] == model]
    is_thinking = model in thinking_models
    
    ax.scatter(model_data['Cost per Sample'], model_data['Accuracy'],
              s=100, alpha=0.6,
              marker='o' if is_thinking else 's',
              label=f"{model} ({'Thinking' if is_thinking else 'Standard'})")

ax.set_xlabel('Cost per Sample ($)')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs Cost Tradeoff')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)

# 2. Token Usage Comparison
ax = axes[0, 1]
model_names = tokenomics_df['Model'].unique()
avg_tokens = tokenomics_df.groupby('Model')['Avg Output Tokens'].mean()

colors = ['coral' if m in thinking_models else 'skyblue' for m in model_names]
bars = ax.bar(range(len(model_names)), avg_tokens.values, color=colors)

ax.set_xlabel('Model')
ax.set_ylabel('Average Output Tokens')
ax.set_title('Token Usage: Thinking vs Standard Models')
ax.set_xticks(range(len(model_names)))
ax.set_xticklabels([m.split(':')[0] for m in model_names], rotation=45)

# Add value labels on bars
for bar, val in zip(bars, avg_tokens.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
           f'{val:.0f}', ha='center', va='bottom')

# 3. ROI Analysis (Accuracy per Dollar)
ax = axes[0, 2]
roi_by_variant = tokenomics_df.pivot_table(
    values='Accuracy per Dollar',
    index='Variant',
    columns='Model',
    aggfunc='mean'
)

roi_by_variant.plot(kind='bar', ax=ax)
ax.set_xlabel('Variant')
ax.set_ylabel('Accuracy per Dollar')
ax.set_title('Return on Investment by Variant')
ax.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.tick_params(axis='x', rotation=45)

# 4. Cost Breakdown by Model Type
ax = axes[1, 0]
thinking_avg_cost = tokenomics_df[tokenomics_df['Model'].isin(thinking_models)]['Cost per Sample'].mean()
standard_avg_cost = tokenomics_df[tokenomics_df['Model'].isin(standard_models)]['Cost per Sample'].mean()

bars = ax.bar(['Thinking Models', 'Standard Models'], 
              [thinking_avg_cost, standard_avg_cost],
              color=['coral', 'skyblue'])

ax.set_ylabel('Average Cost per Sample ($)')
ax.set_title('Cost Comparison: Model Types')

# Add percentage difference
if standard_avg_cost > 0:
    pct_increase = ((thinking_avg_cost - standard_avg_cost) / standard_avg_cost) * 100
    ax.text(0.5, max(thinking_avg_cost, standard_avg_cost) * 1.1,
           f'Thinking models cost {pct_increase:.1f}% more',
           ha='center', fontsize=10, style='italic')

# 5. Accuracy Improvement vs Cost Increase
ax = axes[1, 1]
if len(thinking_models) > 0 and len(standard_models) > 0:
    thinking_avg_acc = tokenomics_df[tokenomics_df['Model'].isin(thinking_models)]['Accuracy'].mean()
    standard_avg_acc = tokenomics_df[tokenomics_df['Model'].isin(standard_models)]['Accuracy'].mean()
    
    acc_improvement = thinking_avg_acc - standard_avg_acc
    cost_increase = thinking_avg_cost - standard_avg_cost
    
    categories = ['Accuracy\nImprovement', 'Cost\nIncrease']
    values = [acc_improvement * 100, cost_increase * 1000]  # Scale for visibility
    colors = ['green' if v > 0 else 'red' for v in values]
    
    bars = ax.bar(categories, np.abs(values), color=colors, alpha=0.7)
    ax.set_ylabel('Magnitude')
    ax.set_title('Thinking Models: Benefits vs Costs')
    ax.set_ylim(0, max(np.abs(values)) * 1.2)
    
    # Add value labels
    for bar, val, orig in zip(bars, values, [acc_improvement, cost_increase]):
        label = f'+{orig:.1%}' if categories[bars.index(bar)] == 'Accuracy\nImprovement' else f'+${orig:.4f}'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
               label, ha='center', va='bottom')

# 6. Efficiency Frontier
ax = axes[1, 2]
# Group by model and calculate mean accuracy and cost
model_summary = tokenomics_df.groupby('Model').agg({
    'Accuracy': 'mean',
    'Cost per Sample': 'mean'
}).reset_index()

# Separate by model type
for is_thinking in [True, False]:
    model_type = 'Thinking' if is_thinking else 'Standard'
    model_list = thinking_models if is_thinking else standard_models
    subset = model_summary[model_summary['Model'].isin(model_list)]
    
    if not subset.empty:
        # Sort by cost for frontier line
        subset_sorted = subset.sort_values('Cost per Sample')
        ax.plot(subset_sorted['Cost per Sample'], subset_sorted['Accuracy'],
               'o-', linewidth=2, markersize=8,
               label=f'{model_type} Models', alpha=0.7)
        
        # Annotate points
        for _, row in subset.iterrows():
            ax.annotate(row['Model'].split(':')[0],
                       (row['Cost per Sample'], row['Accuracy']),
                       xytext=(5, 5), textcoords='offset points',
                       fontsize=8)

ax.set_xlabel('Cost per Sample ($)')
ax.set_ylabel('Accuracy')
ax.set_title('Efficiency Frontier: Optimal Cost-Accuracy Tradeoff')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('SysEngBench Tokenomics Analysis', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_BASE_DIR / 'tokenomics_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## 7. Position Bias Analysis with Cost Considerations

# %%
# Analyze position bias for all models
position_analysis = []

for model in tokenomics_df['Model'].unique():
    mcq_variants = ['sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d']
    
    # Get accuracies and costs for each position
    position_data = {}
    for variant in mcq_variants:
        variant_data = tokenomics_df[(tokenomics_df['Model'] == model) & 
                                     (tokenomics_df['Variant'] == variant)]
        if not variant_data.empty:
            position = variant[-1].upper()
            position_data[position] = {
                'accuracy': variant_data['Accuracy'].values[0],
                'cost': variant_data['Cost per Sample'].values[0],
                'tokens': variant_data['Avg Output Tokens'].values[0]
            }
    
    if len(position_data) == 4:
        accuracies = [position_data[p]['accuracy'] for p in ['A', 'B', 'C', 'D']]
        costs = [position_data[p]['cost'] for p in ['A', 'B', 'C', 'D']]
        tokens = [position_data[p]['tokens'] for p in ['A', 'B', 'C', 'D']]
        
        position_analysis.append({
            'Model': model,
            'Is_Thinking': model in thinking_models,
            'Position_A': accuracies[0],
            'Position_B': accuracies[1],
            'Position_C': accuracies[2],
            'Position_D': accuracies[3],
            'Accuracy_StdDev': np.std(accuracies),
            'Accuracy_Range': max(accuracies) - min(accuracies),
            'Cost_StdDev': np.std(costs),
            'Token_StdDev': np.std(tokens),
            'Avg_Cost': np.mean(costs),
            'Position_Bias_Cost': (np.std(accuracies) / np.mean(accuracies)) * np.mean(costs) if np.mean(accuracies) > 0 else 0
        })

position_bias_df = pd.DataFrame(position_analysis)

print("\n📊 POSITION BIAS ANALYSIS WITH COST IMPACT")
print("="*60)
print(position_bias_df.to_string(index=False))

# %% [markdown]
# ## 8. Original vs Variant Performance

# %%
# Compare original SysEngBench to position-specific variants
original_comparison = []

for model in tokenomics_df['Model'].unique():
    # Get original SysEngBench performance
    original = tokenomics_df[(tokenomics_df['Model'] == model) & 
                             (tokenomics_df['Variant'] == 'sysengbench')]
    
    if not original.empty:
        orig_acc = original['Accuracy'].values[0]
        orig_cost = original['Cost per Sample'].values[0]
        
        # Get average of position variants
        variants = tokenomics_df[(tokenomics_df['Model'] == model) & 
                                 (tokenomics_df['Variant'].str.contains('sysengbench-[abcd]'))]
        
        if not variants.empty:
            var_avg_acc = variants['Accuracy'].mean()
            var_avg_cost = variants['Cost per Sample'].mean()
            
            original_comparison.append({
                'Model': model,
                'Original_Accuracy': orig_acc,
                'Variants_Avg_Accuracy': var_avg_acc,
                'Accuracy_Diff': var_avg_acc - orig_acc,
                'Original_Cost': orig_cost,
                'Variants_Avg_Cost': var_avg_cost,
                'Cost_Diff': var_avg_cost - orig_cost
            })

original_comp_df = pd.DataFrame(original_comparison)

print("\n📈 ORIGINAL vs POSITION VARIANTS COMPARISON")
print("="*60)
print(original_comp_df.to_string(index=False))

# %% [markdown]
# ## 9. Cost-Benefit Recommendations

# %%
def generate_recommendations(tokenomics_df, position_bias_df):
    """Generate cost-benefit recommendations based on analysis"""
    
    recommendations = []
    
    # 1. Analyze thinking vs standard models
    thinking_data = tokenomics_df[tokenomics_df['Model'].isin(thinking_models)]
    standard_data = tokenomics_df[tokenomics_df['Model'].isin(standard_models)]
    
    if not thinking_data.empty and not standard_data.empty:
        thinking_avg_acc = thinking_data['Accuracy'].mean()
        standard_avg_acc = standard_data['Accuracy'].mean()
        thinking_avg_cost = thinking_data['Cost per Sample'].mean()
        standard_avg_cost = standard_data['Cost per Sample'].mean()
        
        acc_improvement = (thinking_avg_acc - standard_avg_acc) / standard_avg_acc * 100
        cost_increase = (thinking_avg_cost - standard_avg_cost) / standard_avg_cost * 100
        
        if acc_improvement > cost_increase:
            recommendations.append({
                'Category': 'Model Selection',
                'Finding': f'Thinking models provide {acc_improvement:.1f}% accuracy improvement for {cost_increase:.1f}% cost increase',
                'Recommendation': '✅ WORTH IT: Accuracy gains exceed cost increases',
                'ROI': f'{(acc_improvement / cost_increase):.2f}x return on investment'
            })
        else:
            recommendations.append({
                'Category': 'Model Selection',
                'Finding': f'Thinking models provide {acc_improvement:.1f}% accuracy improvement for {cost_increase:.1f}% cost increase',
                'Recommendation': '⚠️ SITUATIONAL: Cost increase exceeds accuracy gains',
                'ROI': f'{(acc_improvement / cost_increase):.2f}x return on investment'
            })
    
    # 2. Position bias cost impact
    for _, row in position_bias_df.iterrows():
        if row['Accuracy_Range'] > 0.1:
            wasted_cost = row['Position_Bias_Cost']
            recommendations.append({
                'Category': 'Position Bias',
                'Finding': f"{row['Model']} has {row['Accuracy_Range']:.1%} accuracy range across positions",
                'Recommendation': f'Rotate answer positions to avoid ${wasted_cost:.6f} per sample in bias cost',
                'ROI': 'Immediate savings with no accuracy loss'
            })
    
    # 3. Format recommendations (MCQ vs OSQ)
    for model in tokenomics_df['Model'].unique():
        mcq_data = tokenomics_df[(tokenomics_df['Model'] == model) & 
                                 (tokenomics_df['Variant'].str.contains('sysengbench-[abcd]|sysengbench$'))]
        osq_data = tokenomics_df[(tokenomics_df['Model'] == model) & 
                                (tokenomics_df['Variant'] == 'sysengbench-osq')]
        
        if not mcq_data.empty and not osq_data.empty:
            mcq_acc = mcq_data['Accuracy'].mean()
            osq_acc = osq_data['Accuracy'].mean()
            mcq_cost = mcq_data['Cost per Sample'].mean()
            osq_cost = osq_data['Cost per Sample'].mean()
            
            if osq_acc > mcq_acc and osq_cost < mcq_cost:
                recommendations.append({
                    'Category': 'Format Selection',
                    'Finding': f'{model}: OSQ format has {(osq_acc-mcq_acc):.1%} better accuracy at {(mcq_cost-osq_cost)/mcq_cost:.1%} lower cost',
                    'Recommendation': '🎯 Use OSQ format for this model',
                    'ROI': 'Win-win: Better accuracy AND lower cost'
                })
    
    # 4. Optimal configurations
    best_value = tokenomics_df.loc[tokenomics_df['Accuracy per Dollar'].idxmax()]
    recommendations.append({
        'Category': 'Best Value',
        'Finding': f"{best_value['Model']} on {best_value['Variant']} provides highest accuracy per dollar",
        'Recommendation': f"Use for cost-sensitive applications: {best_value['Accuracy per Dollar']:.0f} accuracy/$ ",
        'ROI': f"{best_value['Accuracy']:.1%} accuracy at ${best_value['Cost per Sample']:.6f}/sample"
    })
    
    # 5. High-accuracy requirements
    best_accuracy = tokenomics_df.loc[tokenomics_df['Accuracy'].idxmax()]
    recommendations.append({
        'Category': 'Maximum Accuracy',
        'Finding': f"{best_accuracy['Model']} on {best_accuracy['Variant']} achieves highest accuracy",
        'Recommendation': f"Use for critical applications: {best_accuracy['Accuracy']:.1%} accuracy",
        'ROI': f"Premium cost of ${best_accuracy['Cost per Sample']:.6f}/sample justified for critical use"
    })
    
    return pd.DataFrame(recommendations)

recommendations_df = generate_recommendations(tokenomics_df, position_bias_df)

print("\n💡 COST-BENEFIT RECOMMENDATIONS")
print("="*80)
for _, rec in recommendations_df.iterrows():
    print(f"\n{rec['Category'].upper()}")
    print(f"  Finding: {rec['Finding']}")
    print(f"  → {rec['Recommendation']}")
    print(f"  ROI: {rec['ROI']}")

# %% [markdown]
# ## 10. Token Efficiency Analysis

# %%
# Analyze token efficiency - accuracy gained per token generated
tokenomics_df['Accuracy_per_1k_Tokens'] = tokenomics_df['Accuracy'] / (tokenomics_df['Avg Output Tokens'] / 1000)
tokenomics_df['Tokens_per_Accuracy_Point'] = tokenomics_df['Avg Output Tokens'] / (tokenomics_df['Accuracy'] * 100)

efficiency_summary = tokenomics_df.groupby('Model').agg({
    'Accuracy_per_1k_Tokens': 'mean',
    'Tokens_per_Accuracy_Point': 'mean',
    'Avg Output Tokens': 'mean',
    'Accuracy': 'mean'
}).round(2)

print("\n⚡ TOKEN EFFICIENCY ANALYSIS")
print("="*60)
print(efficiency_summary.to_string())

# Visualize token efficiency
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. Accuracy per 1k tokens
ax = axes[0]
efficiency_summary['Accuracy_per_1k_Tokens'].plot(kind='bar', ax=ax, color=['coral' if m in thinking_models else 'skyblue' for m in efficiency_summary.index])
ax.set_xlabel('Model')
ax.set_ylabel('Accuracy per 1k Tokens')
ax.set_title('Token Efficiency: Accuracy Gained per 1000 Tokens')
ax.tick_params(axis='x', rotation=45)

# 2. Tokens needed per accuracy point
ax = axes[1]
efficiency_summary['Tokens_per_Accuracy_Point'].plot(kind='bar', ax=ax, color=['coral' if m in thinking_models else 'skyblue' for m in efficiency_summary.index])
ax.set_xlabel('Model')
ax.set_ylabel('Tokens per Accuracy Point')
ax.set_title('Token Cost: Tokens Required per 1% Accuracy')
ax.tick_params(axis='x', rotation=45)

plt.suptitle('Token Efficiency Analysis', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_BASE_DIR / 'token_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## 11. Break-Even Analysis

# %%
def calculate_breakeven_analysis(tokenomics_df):
    """
    Calculate at what volume thinking models become cost-effective
    """
    thinking_data = tokenomics_df[tokenomics_df['Model'].isin(thinking_models)]
    standard_data = tokenomics_df[tokenomics_df['Model'].isin(standard_models)]
    
    if thinking_data.empty or standard_data.empty:
        return None
    
    # Average metrics
    thinking_acc = thinking_data['Accuracy'].mean()
    standard_acc = standard_data['Accuracy'].mean()
    thinking_cost = thinking_data['Cost per Sample'].mean()
    standard_cost = standard_data['Cost per Sample'].mean()
    
    # Calculate break-even scenarios
    volumes = [100, 1000, 10000, 100000, 1000000]
    
    analysis = []
    for volume in volumes:
        # Cost at volume
        thinking_total = thinking_cost * volume
        standard_total = standard_cost * volume
        
        # Expected correct answers
        thinking_correct = thinking_acc * volume
        standard_correct = standard_acc * volume
        
        # Cost per correct answer
        thinking_per_correct = thinking_total / thinking_correct if thinking_correct > 0 else float('inf')
        standard_per_correct = standard_total / standard_correct if standard_correct > 0 else float('inf')
        
        # Value calculation (assuming each correct answer has business value)
        additional_correct = thinking_correct - standard_correct
        additional_cost = thinking_total - standard_total
        
        analysis.append({
            'Volume': volume,
            'Thinking_Total_Cost': thinking_total,
            'Standard_Total_Cost': standard_total,
            'Additional_Cost': additional_cost,
            'Thinking_Correct': int(thinking_correct),
            'Standard_Correct': int(standard_correct),
            'Additional_Correct': int(additional_correct),
            'Cost_per_Additional_Correct': additional_cost / additional_correct if additional_correct > 0 else float('inf'),
            'Thinking_Cost_per_Correct': thinking_per_correct,
            'Standard_Cost_per_Correct': standard_per_correct
        })
    
    return pd.DataFrame(analysis)

breakeven_df = calculate_breakeven_analysis(tokenomics_df)

if breakeven_df is not None:
    print("\n💰 BREAK-EVEN ANALYSIS: When Do Thinking Models Pay Off?")
    print("="*80)
    print(breakeven_df.to_string(index=False))
    
    # Find break-even point
    print("\n📊 INSIGHTS:")
    print("-"*40)
    
    # Calculate value threshold
    avg_cost_per_additional = breakeven_df['Cost_per_Additional_Correct'].mean()
    print(f"Average cost per additional correct answer: ${avg_cost_per_additional:.4f}")
    print(f"→ Thinking models are worth it if each additional correct answer is worth >${avg_cost_per_additional:.4f}")
    
    # ROI at different scales
    for _, row in breakeven_df.iterrows():
        if row['Additional_Correct'] > 0:
            roi_multiplier = row['Additional_Correct'] / row['Additional_Cost'] * 100
            print(f"\nAt {row['Volume']:,} samples:")
            print(f"  - Extra cost: ${row['Additional_Cost']:.2f}")
            print(f"  - Extra correct: {row['Additional_Correct']}")
            print(f"  - ROI: {roi_multiplier:.1f} correct answers per $100 spent")



# Summary statistics
summary_stats = {
    'Total Evaluations': len(tokenomics_df),
    'Models Tested': tokenomics_df['Model'].nunique(),
    'Variants Tested': tokenomics_df['Variant'].nunique(),
    'Total Cost': tokenomics_df['Total Cost'].sum(),
    'Total Samples': tokenomics_df['Total Cost'].count() * LIMIT if LIMIT else 'Full dataset',
    'Average Accuracy (Thinking)': thinking_data['Accuracy'].mean() if 'thinking_data' in locals() else 'N/A',
    'Average Accuracy (Standard)': standard_data['Accuracy'].mean() if 'standard_data' in locals() else 'N/A',
    'Best Value Model': tokenomics_df.loc[tokenomics_df['Accuracy per Dollar'].idxmax(), 'Model'],
    'Best Accuracy Model': tokenomics_df.loc[tokenomics_df['Accuracy'].idxmax(), 'Model']
}

with open(OUTPUT_BASE_DIR / 'summary_statistics.json', 'w') as f:
    json.dump(summary_stats, f, indent=2, default=str)

print("\n📁 All results exported to:", OUTPUT_BASE_DIR)
print("\nFiles created:")
for file in OUTPUT_BASE_DIR.glob('*.csv'):
    print(f"  - {file.name}")
for file in OUTPUT_BASE_DIR.glob('*.png'):
    print(f"  - {file.name}")
for file in OUTPUT_BASE_DIR.glob('*.html'):
    print(f"  - {file.name}")


## Proposed / Notional Tokenomics Summary
Example Insights


Model                Accuracy  Avg_Tokens  Cost/Sample  Acc/$
deepseek-r1:7b       0.78      3,245       $0.00097     803
qwq:32b              0.82      4,512       $0.00271     302  
llama3.2:3b          0.61      187         $0.00002     30,500
mistral:7b           0.65      234         $0.00003     21,667


Ideas for cost analysis:

At 10,000 samples:
- Extra cost: $24.50
- Extra correct: 1,700
- Cost per extra correct: $0.0144
→ Worth it if each correct answer value > $0.0144

# Statistical Analysis

**Reasons for the statistical tests**

Statistical Test Suite:
1. Normality Testing

Shapiro-Wilk (best for small samples)
D'Agostino's K² (skewness and kurtosis)
Jarque-Bera (large sample test)
Q-Q plots to visually assess normality

2. Position Bias Analysis

Chi-Square Test - Overall independence of position and accuracy
Kruskal-Wallis H Test - Non-parametric ANOVA
Friedman Test - For repeated measures
Pairwise McNemar Tests - Between each position pair
Cramér's V - Effect size for categorical data

3. Effect Size Metrics

Cohen's d - Standardized mean difference
Glass's Δ - Uses control group SD
Hedges' g - Small sample correction
Interpretation guidelines (negligible/small/medium/large)

4. Bootstrap Analysis

10,000 resamples for robust confidence intervals
95% CI for all accuracy estimates
CI for differences (e.g., MCQ - OSQ)
Determines if differences are statistically significant

5. Power Analysis

Calculates achieved statistical power
Determines if studies are adequately powered
Calculates required sample sizes for 80% power
Identifies underpowered comparisons

6. Inter-Model Reliability

Cohen's κappa - Agreement between models
Percentage agreement calculations
Interpretation (poor/fair/moderate/substantial/perfect)

7. Regression Analysis

Logistic Regression - Identifies predictors of accuracy
Mixed Effects Models - Accounts for question difficulty
Odds ratios with interpretations
Significance of predictors (thinking model, format, position)

8. Permutation Tests

Distribution-free hypothesis testing
10,000 permutations for p-values
Visualization of null distributions
Robust to non-normal data

9. Clustering Analysis

K-means clustering of questions by difficulty
PCA visualization of question patterns
Identifies groups of similar questions
Helps understand what makes questions hard/easy

10. Bayesian Analysis (if PyMC3 installed)

Posterior distributions for model accuracy
95% and 89% credible intervals
Probability that Model A > Model B
Evidence strength interpretation

11. Response Complexity Analysis

Mann-Whitney U tests for response length differences
Correlation between response length and accuracy
Tests if correct answers are longer/shorter

12. Multiple Comparisons Correction

Bonferroni (most conservative)
Holm (step-down method)
Benjamini-Hochberg (FDR control)
Benjamini-Yekutieli (FDR under dependence)

🔍 Key Statistical Questions Answered:

Is position bias statistically significant?

Chi-square and Friedman tests confirm
Effect sizes quantify practical importance


Are differences between models real or due to chance?

Bootstrap CIs show if differences are significant
Permutation tests provide exact p-values


Do we have enough data for reliable conclusions?

Power analysis shows if sample size is adequate
Calculates how many more samples needed


Which factors predict accuracy?

Regression identifies significant predictors
Odds ratios show magnitude of effects


Are results consistent across questions?

Clustering identifies question types
Inter-rater reliability shows model agreement



📈 Interpretation Guidelines:
The notebook provides clear interpretations:
python# Effect Sizes
Cohen's d < 0.2: Negligible
0.2-0.5: Small
0.5-0.8: Medium
> 0.8: Large

**Statistical Power**
< 0.8: Underpowered (need more data)
≥ 0.8: Adequately powered

**P-values (after correction)**
< 0.001: Very strong evidence (***)
< 0.01: Strong evidence (**)
< 0.05: Moderate evidence (*)
≥ 0.05: No significant evidence
🎯 Practical Applications:

Publication-Ready Statistics - All tests needed for academic papers
Decision Support - Statistical evidence for model selection
Bias Detection - Quantifies and tests for various biases
Sample Size Planning - Determines data needs for future studies
Robust Conclusions - Multiple tests confirm findings

In [ ]:
from scipy import stats

# Statistical tests for position bias
print("\n📈 Statistical Analysis:")
print("-" * 50)

for model in analyzer.all_results.keys():
    print(f"\n{model}:")
    
    # Get accuracies for each position
    accuracies = []
    labels = []
    for variant in ["sysengbench-a", "sysengbench-b", "sysengbench-c", "sysengbench-d"]:
        if variant in analyzer.all_results[model]:
            samples = analyzer.all_results[model][variant]["samples"]
            correct = [1 if s['target'] == s['pred'] else 0 for s in samples]
            accuracies.append(correct)
            labels.append(variant[-1].upper())
    
    if len(accuracies) == 4:
        # Chi-square test for independence
        # Create contingency table
        contingency = []
        for acc_list in accuracies:
            correct = sum(acc_list)
            incorrect = len(acc_list) - correct
            contingency.append([correct, incorrect])
        
        chi2, p_value = stats.chi2_contingency(contingency)[:2]
        print(f"  Chi-square test for position independence: p-value = {p_value:.4f}")
        
        if p_value < 0.05:
            print(f"  ⚠️ Significant position bias detected (p < 0.05)")
        else:
            print(f"  ✅ No significant position bias detected (p >= 0.05)")
        
        # ANOVA test
        f_stat, anova_p = stats.f_oneway(*accuracies)
        print(f"  ANOVA test: F-statistic = {f_stat:.4f}, p-value = {anova_p:.4f}")


In [ ]:
from scipy import stats
import itertools

def statistical_significance_test(analyzer):
    """
    Perform statistical significance tests for position bias and format differences
    """
    print("\n" + "="*60)
    print("STATISTICAL SIGNIFICANCE TESTING")
    print("="*60)
    
    for model in analyzer.all_results.keys():
        print(f"\n📊 Model: {model}")
        print("-"*40)
        
        # 1. Test for position bias (Chi-square test)
        position_samples = {}
        for variant in ["sysengbench-a", "sysengbench-b", "sysengbench-c", "sysengbench-d"]:
            if variant in analyzer.all_results[model]:
                samples = analyzer.all_results[model][variant]["samples"]
                position_samples[variant[-1].upper()] = [
                    1 if s['target'] == s['pred'] else 0 for s in samples
                ]
        
        if len(position_samples) == 4:
            # Perform Chi-square test
            contingency_table = []
            for position, results in position_samples.items():
                correct = sum(results)
                incorrect = len(results) - correct
                contingency_table.append([correct, incorrect])
            
            chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)
            
            print(f"Position Bias Chi-Square Test:")
            print(f"  Chi-square statistic: {chi2:.4f}")
            print(f"  P-value: {p_value:.4f}")
            print(f"  Degrees of freedom: {dof}")
            
            if p_value < 0.05:
                print(f"  ⚠️ SIGNIFICANT position bias detected (p < 0.05)")
            else:
                print(f"  ✅ No significant position bias (p >= 0.05)")
            
            # Pairwise comparisons
            print(f"\nPairwise Position Comparisons (McNemar's test):")
            positions = list(position_samples.keys())
            for pos1, pos2 in itertools.combinations(positions, 2):
                # McNemar's test for paired samples
                data1 = position_samples[pos1]
                data2 = position_samples[pos2]
                
                # Create contingency table for McNemar's test
                both_correct = sum(1 for i in range(len(data1)) if data1[i] == 1 and data2[i] == 1)
                only1_correct = sum(1 for i in range(len(data1)) if data1[i] == 1 and data2[i] == 0)
                only2_correct = sum(1 for i in range(len(data1)) if data1[i] == 0 and data2[i] == 1)
                both_wrong = sum(1 for i in range(len(data1)) if data1[i] == 0 and data2[i] == 0)
                
                # Perform McNemar's test
                if only1_correct + only2_correct > 0:
                    statistic = (abs(only1_correct - only2_correct) - 1)**2 / (only1_correct + only2_correct)
                    p_val = 1 - stats.chi2.cdf(statistic, 1)
                    
                    print(f"  {pos1} vs {pos2}: p={p_val:.4f}", end="")
                    if p_val < 0.05:
                        print(" *")
                    else:
                        print()
        
        # 2. Test MCQ vs OSQ difference
        if "sysengbench-osq" in analyzer.all_results[model]:
            print(f"\nMCQ vs OSQ Comparison:")
            
            # Get MCQ combined results
            mcq_results = []
            for variant in ["sysengbench-a", "sysengbench-b", "sysengbench-c", "sysengbench-d"]:
                if variant in analyzer.all_results[model]:
                    samples = analyzer.all_results[model][variant]["samples"]
                    mcq_results.extend([1 if s['target'] == s['pred'] else 0 for s in samples])
            
            # Get OSQ results
            osq_samples = analyzer.all_results[model]["sysengbench-osq"]["samples"]
            osq_results = [1 if s['target'] == s['pred'] else 0 for s in osq_samples]
            
            if mcq_results and osq_results:
                # Two-proportion z-test
                mcq_acc = sum(mcq_results) / len(mcq_results)
                osq_acc = sum(osq_results) / len(osq_results)
                
                pooled_p = (sum(mcq_results) + sum(osq_results)) / (len(mcq_results) + len(osq_results))
                se = np.sqrt(pooled_p * (1 - pooled_p) * (1/len(mcq_results) + 1/len(osq_results)))
                
                if se > 0:
                    z_stat = (mcq_acc - osq_acc) / se
                    p_val = 2 * (1 - stats.norm.cdf(abs(z_stat)))
                    
                    print(f"  MCQ Accuracy: {mcq_acc:.3f}")
                    print(f"  OSQ Accuracy: {osq_acc:.3f}")
                    print(f"  Z-statistic: {z_stat:.4f}")
                    print(f"  P-value: {p_val:.4f}")
                    
                    if p_val < 0.05:
                        better = "MCQ" if mcq_acc > osq_acc else "OSQ"
                        print(f"  ⚠️ SIGNIFICANT difference - {better} performs better")
                    else:
                        print(f"  ✅ No significant difference between formats")

# Run statistical tests
statistical_significance_test(analyzer)

# More Detailed Statistical Analysis (Merge with Above)

Pick and choose what we want here.

## Setup and Imports

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Statistical imports
from scipy import stats
from scipy.stats import (
    chi2_contingency, fisher_exact, mcnemar,
    wilcoxon, mannwhitneyu, kruskal, friedmanchisquare,
    shapiro, normaltest, jarque_bera,
    ttest_ind, ttest_rel, f_oneway,
    pearsonr, spearmanr, kendalltau,
    binom_test, multinomial
)
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_sm
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.power import TTestPower, FTestPower
from statsmodels.stats.inter_rater import fleiss_kappa, cohens_kappa
import statsmodels.api as sm
from sklearn.metrics import cohen_kappa_score, matthews_corrcoef
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# For bootstrap and permutation tests
from scipy.stats import bootstrap
import numpy.random as npr

# For Bayesian analysis
try:
    import pymc3 as pm
    BAYESIAN_AVAILABLE = True
except ImportError:
    BAYESIAN_AVAILABLE = False
    print("PyMC3 not available - Bayesian analysis will be skipped")

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


## Normality tests.. (Double Check if we want)

In [ ]:

# %% [markdown]
# ## 3. Normality Testing

# %%
def test_normality(data, name="Data"):
    """
    Test for normality using multiple methods
    """
    results = {}
    
    # Shapiro-Wilk test (best for small samples)
    if len(data) < 5000:
        stat, p_shapiro = shapiro(data)
        results['Shapiro-Wilk'] = {'statistic': stat, 'p_value': p_shapiro}
    
    # D'Agostino's K-squared test
    stat, p_dagostino = normaltest(data)
    results["D'Agostino's K²"] = {'statistic': stat, 'p_value': p_dagostino}
    
    # Jarque-Bera test
    stat, p_jb = jarque_bera(data)
    results['Jarque-Bera'] = {'statistic': stat, 'p_value': p_jb}
    
    # Q-Q plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Histogram with normal overlay
    ax = axes[0]
    ax.hist(data, bins=30, density=True, alpha=0.7, edgecolor='black')
    mu, std = np.mean(data), np.std(data)
    x = np.linspace(data.min(), data.max(), 100)
    ax.plot(x, stats.norm.pdf(x, mu, std), 'r-', lw=2, label='Normal')
    ax.set_title(f'{name} Distribution')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend()
    
    # Q-Q plot
    ax = axes[1]
    stats.probplot(data, dist="norm", plot=ax)
    ax.set_title(f'{name} Q-Q Plot')
    
    plt.tight_layout()
    plt.show()
    
    # Print results
    print(f"\n📊 Normality Tests for {name}:")
    print("-" * 50)
    for test_name, test_results in results.items():
        p_val = test_results['p_value']
        is_normal = p_val > 0.05
        print(f"{test_name:20} p={p_val:.4f} {'✓ Normal' if is_normal else '✗ Not Normal'}")
    
    return results

# Test normality of accuracy by model
for model in df['model'].unique():
    model_accuracies = df[df['model'] == model].groupby('variant')['correct'].mean().values
    if len(model_accuracies) > 3:
        test_normality(model_accuracies, f"{model} Accuracies")

# SysEngBench Comprehensive Statistical Analysis
 
This notebook performs extensive statistical analysis including:
1. Parametric and non-parametric tests
2. Effect size calculations
3. Power analysis
4. Bayesian analysis
5. Time series analysis (if multiple runs)
6. Clustering and pattern detection
7. Regression analysis
8. Bootstrap confidence intervals

## Position Bias Statistical Tests - Need to adjudicate with the one in Analys 2 Section

In [ ]:
def analyze_position_bias_statistics(df):
    """
    Comprehensive statistical analysis of position bias
    """
    results = {}
    
    for model in df['model'].unique():
        print(f"\n{'='*60}")
        print(f"POSITION BIAS ANALYSIS: {model}")
        print('='*60)
        
        # Get data for position variants
        position_data = {}
        for pos in ['a', 'b', 'c', 'd']:
            variant_name = f'sysengbench-{pos}'
            variant_data = df[(df['model'] == model) & (df['variant'] == variant_name)]
            if not variant_data.empty:
                position_data[pos.upper()] = variant_data['correct'].values
        
        if len(position_data) != 4:
            print(f"⚠️ Incomplete position data for {model}")
            continue
        
        # 1. Chi-Square Test for Independence
        contingency = []
        for pos, correct_array in position_data.items():
            contingency.append([sum(correct_array), len(correct_array) - sum(correct_array)])
        
        chi2, p_chi2, dof, expected = chi2_contingency(contingency)
        
        # 2. Kruskal-Wallis H Test (non-parametric ANOVA)
        h_stat, p_kruskal = kruskal(*position_data.values())
        
        # 3. Friedman Test (for repeated measures)
        # Reshape data for Friedman test
        n_questions = min(len(v) for v in position_data.values())
        friedman_data = np.array([v[:n_questions] for v in position_data.values()]).T
        if friedman_data.shape[0] > 2:
            f_stat, p_friedman = friedmanchisquare(*friedman_data.T)
        else:
            p_friedman = np.nan
        
        # 4. Effect Size (Cramér's V)
        n = sum(len(v) for v in position_data.values())
        cramers_v = np.sqrt(chi2 / (n * (min(len(contingency), len(contingency[0])) - 1)))
        
        # 5. Pairwise McNemar Tests
        pairwise_results = {}
        positions = list(position_data.keys())
        for i in range(len(positions)):
            for j in range(i+1, len(positions)):
                pos1, pos2 = positions[i], positions[j]
                data1, data2 = position_data[pos1][:n_questions], position_data[pos2][:n_questions]
                
                # Create contingency table for McNemar
                both_correct = sum((data1[k] == 1) & (data2[k] == 1) for k in range(len(data1)))
                only1_correct = sum((data1[k] == 1) & (data2[k] == 0) for k in range(len(data1)))
                only2_correct = sum((data1[k] == 0) & (data2[k] == 1) for k in range(len(data1)))
                neither_correct = sum((data1[k] == 0) & (data2[k] == 0) for k in range(len(data1)))
                
                table = [[both_correct, only1_correct], 
                        [only2_correct, neither_correct]]
                
                result = mcnemar_sm(table, exact=True)
                pairwise_results[f'{pos1} vs {pos2}'] = {
                    'statistic': result.statistic,
                    'p_value': result.pvalue
                }
        
        # Print results
        print(f"\n📈 Overall Tests:")
        print(f"  Chi-Square Test: χ²={chi2:.3f}, p={p_chi2:.4f}")
        print(f"  Kruskal-Wallis: H={h_stat:.3f}, p={p_kruskal:.4f}")
        if not np.isnan(p_friedman):
            print(f"  Friedman Test: p={p_friedman:.4f}")
        print(f"  Effect Size (Cramér's V): {cramers_v:.3f}")
        
        # Interpret effect size
        if cramers_v < 0.1:
            effect_interpretation = "Negligible"
        elif cramers_v < 0.3:
            effect_interpretation = "Small"
        elif cramers_v < 0.5:
            effect_interpretation = "Medium"
        else:
            effect_interpretation = "Large"
        print(f"  Effect Size Interpretation: {effect_interpretation}")
        
        print(f"\n📊 Pairwise Comparisons (McNemar):")
        for pair, result in pairwise_results.items():
            sig = "*" if result['p_value'] < 0.05 else ""
            print(f"  {pair}: p={result['p_value']:.4f} {sig}")
        
        # Store results
        results[model] = {
            'chi2': {'stat': chi2, 'p': p_chi2},
            'kruskal': {'stat': h_stat, 'p': p_kruskal},
            'friedman': {'p': p_friedman},
            'cramers_v': cramers_v,
            'pairwise': pairwise_results
        }
    
    return results

position_bias_results = analyze_position_bias_statistics(df)

## Bootstrap Confidence Intervals

In [ ]:

def bootstrap_confidence_intervals(df, n_bootstrap=10000):
    """
    Calculate bootstrap confidence intervals for accuracy differences
    """
    print("\n" + "="*60)
    print("BOOTSTRAP CONFIDENCE INTERVALS (95%)")
    print("="*60)
    
    results = {}
    
    for model in df['model'].unique():
        print(f"\n📊 {model}:")
        model_results = {}
        
        # Get accuracies by variant
        variants = df[df['model'] == model]['variant'].unique()
        
        for variant in variants:
            variant_data = df[(df['model'] == model) & (df['variant'] == variant)]['correct'].values
            
            if len(variant_data) > 0:
                # Bootstrap for mean accuracy
                def mean_stat(x, axis):
                    return np.mean(x, axis=axis)
                
                # Create bootstrap distribution
                rng = np.random.default_rng(42)
                res = bootstrap(
                    (variant_data,),
                    mean_stat,
                    n_resamples=n_bootstrap,
                    random_state=rng,
                    method='percentile'
                )
                
                ci_lower = res.confidence_interval.low
                ci_upper = res.confidence_interval.high
                point_estimate = np.mean(variant_data)
                
                model_results[variant] = {
                    'mean': point_estimate,
                    'ci_lower': ci_lower,
                    'ci_upper': ci_upper,
                    'ci_width': ci_upper - ci_lower
                }
                
                print(f"  {variant:20} {point_estimate:.3f} [{ci_lower:.3f}, {ci_upper:.3f}]")
        
        # Compare MCQ vs OSQ if available
        mcq_accs = []
        for v in ['sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d']:
            if v in model_results:
                data = df[(df['model'] == model) & (df['variant'] == v)]['correct'].values
                mcq_accs.extend(data)
        
        osq_data = df[(df['model'] == model) & (df['variant'] == 'sysengbench-osq')]['correct'].values
        
        if len(mcq_accs) > 0 and len(osq_data) > 0:
            # Bootstrap difference
            def diff_means(x, y, axis):
                return np.mean(x, axis=axis) - np.mean(y, axis=axis)
            
            rng = np.random.default_rng(42)
            res_diff = bootstrap(
                (np.array(mcq_accs), osq_data),
                lambda x, y, axis: np.mean(x, axis=axis) - np.mean(y, axis=axis),
                n_resamples=n_bootstrap,
                random_state=rng,
                method='percentile'
            )
            
            diff_ci_lower = res_diff.confidence_interval.low
            diff_ci_upper = res_diff.confidence_interval.high
            diff_mean = np.mean(mcq_accs) - np.mean(osq_data)
            
            print(f"\n  MCQ - OSQ difference: {diff_mean:.3f} [{diff_ci_lower:.3f}, {diff_ci_upper:.3f}]")
            
            if diff_ci_lower > 0:
                print("  → MCQ significantly better (CI doesn't include 0)")
            elif diff_ci_upper < 0:
                print("  → OSQ significantly better (CI doesn't include 0)")
            else:
                print("  → No significant difference (CI includes 0)")
        
        results[model] = model_results
    
    return results

bootstrap_results = bootstrap_confidence_intervals(df)

## Effect Size Calculations

In [ ]:

def calculate_effect_sizes(df):
    """
    Calculate various effect size measures
    """
    print("\n" + "="*60)
    print("EFFECT SIZE ANALYSIS")
    print("="*60)
    
    results = {}
    
    for model in df['model'].unique():
        print(f"\n📏 {model}:")
        
        # Cohen's d for pairwise comparisons
        variants = df[df['model'] == model]['variant'].unique()
        
        for i, v1 in enumerate(variants):
            for v2 in variants[i+1:]:
                data1 = df[(df['model'] == model) & (df['variant'] == v1)]['correct'].values
                data2 = df[(df['model'] == model) & (df['variant'] == v2)]['correct'].values
                
                if len(data1) > 0 and len(data2) > 0:
                    # Cohen's d
                    mean1, mean2 = np.mean(data1), np.mean(data2)
                    pooled_std = np.sqrt((np.var(data1) + np.var(data2)) / 2)
                    
                    if pooled_std > 0:
                        cohens_d = (mean1 - mean2) / pooled_std
                        
                        # Interpret Cohen's d
                        if abs(cohens_d) < 0.2:
                            interpretation = "Negligible"
                        elif abs(cohens_d) < 0.5:
                            interpretation = "Small"
                        elif abs(cohens_d) < 0.8:
                            interpretation = "Medium"
                        else:
                            interpretation = "Large"
                        
                        print(f"  {v1} vs {v2}: d={cohens_d:.3f} ({interpretation})")
                        
                        # Glass's delta (uses control group SD)
                        glass_delta = (mean1 - mean2) / np.std(data2)
                        
                        # Hedges' g (corrected for small samples)
                        n1, n2 = len(data1), len(data2)
                        correction = 1 - (3 / (4 * (n1 + n2) - 9))
                        hedges_g = cohens_d * correction
                        
                        results[f"{model}_{v1}_vs_{v2}"] = {
                            'cohens_d': cohens_d,
                            'glass_delta': glass_delta,
                            'hedges_g': hedges_g,
                            'interpretation': interpretation
                        }
    
    return results

effect_sizes = calculate_effect_sizes(df)

## Power Analysis

In [ ]:

def perform_power_analysis(df):
    """
    Perform statistical power analysis
    """
    print("\n" + "="*60)
    print("STATISTICAL POWER ANALYSIS")
    print("="*60)
    
    power_analysis = TTestPower()
    
    for model in df['model'].unique():
        print(f"\n⚡ {model}:")
        
        # Calculate observed effect sizes
        variants = df[df['model'] == model]['variant'].unique()
        
        for v1, v2 in [('sysengbench-a', 'sysengbench-b'), 
                       ('sysengbench', 'sysengbench-osq')]:
            if v1 in variants and v2 in variants:
                data1 = df[(df['model'] == model) & (df['variant'] == v1)]['correct'].values
                data2 = df[(df['model'] == model) & (df['variant'] == v2)]['correct'].values
                
                if len(data1) > 0 and len(data2) > 0:
                    # Calculate effect size
                    mean1, mean2 = np.mean(data1), np.mean(data2)
                    pooled_std = np.sqrt((np.var(data1) + np.var(data2)) / 2)
                    
                    if pooled_std > 0:
                        effect_size = abs(mean1 - mean2) / pooled_std
                        n = min(len(data1), len(data2))
                        
                        # Calculate achieved power
                        power = power_analysis.solve_power(
                            effect_size=effect_size,
                            nobs=n,
                            alpha=0.05,
                            alternative='two-sided'
                        )
                        
                        # Required sample size for 0.8 power
                        required_n = power_analysis.solve_power(
                            effect_size=effect_size,
                            power=0.8,
                            alpha=0.05,
                            alternative='two-sided'
                        )
                        
                        print(f"\n  {v1} vs {v2}:")
                        print(f"    Effect size: {effect_size:.3f}")
                        print(f"    Current n: {n}")
                        print(f"    Achieved power: {power:.3f}")
                        print(f"    Required n for 0.8 power: {required_n:.0f}")
                        
                        if power < 0.8:
                            print(f"    ⚠️ Underpowered - need {required_n - n:.0f} more samples")
                        else:
                            print(f"    ✅ Adequately powered")

perform_power_analysis(df)

## 8. Inter-rater Reliability (Between Models)

In [ ]:
def calculate_inter_rater_reliability(df):
    """
    Calculate agreement between different models
    """
    print("\n" + "="*60)
    print("INTER-MODEL RELIABILITY ANALYSIS")
    print("="*60)
    
    models = df['model'].unique()
    
    if len(models) < 2:
        print("Need at least 2 models for inter-rater reliability")
        return
    
    # Prepare data for each variant
    for variant in df['variant'].unique():
        variant_data = df[df['variant'] == variant]
        
        # Get predictions from each model for same questions
        model_predictions = {}
        questions = variant_data['question'].unique()
        
        for model in models:
            model_data = variant_data[variant_data['model'] == model]
            if not model_data.empty:
                preds = []
                for q in questions:
                    q_data = model_data[model_data['question'] == q]
                    if not q_data.empty:
                        preds.append(1 if q_data.iloc[0]['correct'] else 0)
                    else:
                        preds.append(np.nan)
                model_predictions[model] = preds
        
        # Calculate pairwise Cohen's kappa
        if len(model_predictions) >= 2:
            print(f"\n📊 Variant: {variant}")
            
            model_names = list(model_predictions.keys())
            for i in range(len(model_names)):
                for j in range(i+1, len(model_names)):
                    m1, m2 = model_names[i], model_names[j]
                    
                    # Remove NaN pairs
                    pairs = [(model_predictions[m1][k], model_predictions[m2][k]) 
                            for k in range(len(questions))
                            if not (np.isnan(model_predictions[m1][k]) or 
                                   np.isnan(model_predictions[m2][k]))]
                    
                    if pairs:
                        pred1, pred2 = zip(*pairs)
                        kappa = cohen_kappa_score(pred1, pred2)
                        
                        # Calculate percentage agreement
                        agreement = sum(p1 == p2 for p1, p2 in pairs) / len(pairs)
                        
                        # Interpret kappa
                        if kappa < 0:
                            interpretation = "Poor (worse than chance)"
                        elif kappa < 0.20:
                            interpretation = "Slight"
                        elif kappa < 0.40:
                            interpretation = "Fair"
                        elif kappa < 0.60:
                            interpretation = "Moderate"
                        elif kappa < 0.80:
                            interpretation = "Substantial"
                        else:
                            interpretation = "Almost perfect"
                        
                        print(f"  {m1} vs {m2}:")
                        print(f"    Cohen's κ = {kappa:.3f} ({interpretation})")
                        print(f"    Agreement = {agreement:.1%}")

calculate_inter_rater_reliability(df)


## 9. Regression Analysis

In [ ]:

def perform_regression_analysis(df):
    """
    Perform regression analysis to identify predictors of accuracy
    """
    print("\n" + "="*60)
    print("REGRESSION ANALYSIS")
    print("="*60)
    
    # Prepare features
    regression_data = df.copy()
    
    # Create dummy variables
    regression_data['is_thinking_model'] = regression_data['model'].str.contains('r1|qwq', case=False).astype(int)
    regression_data['is_mcq'] = (~regression_data['variant'].str.contains('osq')).astype(int)
    regression_data['position_a'] = (regression_data['variant'] == 'sysengbench-a').astype(int)
    regression_data['position_b'] = (regression_data['variant'] == 'sysengbench-b').astype(int)
    regression_data['position_c'] = (regression_data['variant'] == 'sysengbench-c').astype(int)
    regression_data['position_d'] = (regression_data['variant'] == 'sysengbench-d').astype(int)
    
    # Normalize response length
    regression_data['response_length_norm'] = (regression_data['response_length'] - 
                                               regression_data['response_length'].mean()) / regression_data['response_length'].std()
    
    # Logistic regression
    X = regression_data[['is_thinking_model', 'is_mcq', 'position_a', 
                         'position_b', 'position_c', 'position_d', 
                         'response_length_norm']]
    X = sm.add_constant(X)
    y = regression_data['correct']
    
    # Remove any NaN values
    mask = ~(X.isna().any(axis=1) | y.isna())
    X = X[mask]
    y = y[mask]
    
    # Fit logistic regression
    logit_model = sm.Logit(y, X)
    logit_result = logit_model.fit(disp=0)
    
    print("\n📈 Logistic Regression Results:")
    print(logit_result.summary2())
    
    # Calculate odds ratios
    print("\n📊 Odds Ratios:")
    odds_ratios = np.exp(logit_result.params)
    for var, or_val in odds_ratios.items():
        if var != 'const':
            print(f"  {var:25} OR={or_val:.3f}")
            if or_val > 1:
                print(f"    → Increases odds of correct answer by {(or_val-1)*100:.1f}%")
            else:
                print(f"    → Decreases odds of correct answer by {(1-or_val)*100:.1f}%")
    
    # Mixed effects model (if multiple questions)
    print("\n📈 Mixed Effects Model (accounting for question difficulty):")
    
    # Prepare data for mixed effects
    me_data = regression_data.copy()
    me_data['question_id'] = me_data.groupby('question').ngroup()
    
    # Simple random effects approximation
    question_effects = me_data.groupby('question_id')['correct'].mean()
    me_data['question_difficulty'] = me_data['question_id'].map(question_effects)
    
    X_me = me_data[['is_thinking_model', 'is_mcq', 'question_difficulty']]
    X_me = sm.add_constant(X_me)
    y_me = me_data['correct']
    
    mask = ~(X_me.isna().any(axis=1) | y_me.isna())
    X_me = X_me[mask]
    y_me = y_me[mask]
    
    me_model = sm.Logit(y_me, X_me)
    me_result = me_model.fit(disp=0)
    
    print("\nKey Findings:")
    for var in ['is_thinking_model', 'is_mcq', 'question_difficulty']:
        if var in me_result.params:
            coef = me_result.params[var]
            p_val = me_result.pvalues[var]
            significant = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
            print(f"  {var:20} β={coef:.3f}, p={p_val:.4f} {significant}")
    
    return logit_result, me_result

regression_results = perform_regression_analysis(df)

## 10. Permutation Tests

In [ ]:

def permutation_test(data1, data2, n_permutations=10000):
    """
    Perform permutation test for difference in means
    """
    observed_diff = np.mean(data1) - np.mean(data2)
    combined = np.concatenate([data1, data2])
    n1 = len(data1)
    
    permuted_diffs = []
    rng = np.random.default_rng(42)
    
    for _ in range(n_permutations):
        permuted = rng.permutation(combined)
        perm_diff = np.mean(permuted[:n1]) - np.mean(permuted[n1:])
        permuted_diffs.append(perm_diff)
    
    permuted_diffs = np.array(permuted_diffs)
    p_value = np.mean(np.abs(permuted_diffs) >= np.abs(observed_diff))
    
    return observed_diff, p_value, permuted_diffs

print("\n" + "="*60)
print("PERMUTATION TESTS")
print("="*60)

for model in df['model'].unique():
    print(f"\n🔀 {model}:")
    
    # MCQ vs OSQ
    mcq_data = df[(df['model'] == model) & (df['variant'].str.contains('sysengbench-[abcd]'))]['correct'].values
    osq_data = df[(df['model'] == model) & (df['variant'] == 'sysengbench-osq')]['correct'].values
    
    if len(mcq_data) > 0 and len(osq_data) > 0:
        obs_diff, p_val, perm_diffs = permutation_test(mcq_data, osq_data)
        
        print(f"  MCQ vs OSQ:")
        print(f"    Observed difference: {obs_diff:.4f}")
        print(f"    Permutation p-value: {p_val:.4f}")
        
        # Plot distribution
        plt.figure(figsize=(8, 4))
        plt.hist(perm_diffs, bins=50, density=True, alpha=0.7, edgecolor='black')
        plt.axvline(obs_diff, color='red', linestyle='--', linewidth=2, label=f'Observed ({obs_diff:.4f})')
        plt.xlabel('Difference in Means')
        plt.ylabel('Density')
        plt.title(f'{model}: Permutation Test Distribution')
        plt.legend()
        plt.show()

## 11. Clustering Analysis

In [ ]:

def perform_clustering_analysis(df):
    """
    Cluster questions based on model performance patterns
    """
    print("\n" + "="*60)
    print("QUESTION CLUSTERING ANALYSIS")
    print("="*60)
    
    # Create question-model accuracy matrix
    questions = df['question'].unique()
    models = df['model'].unique()
    
    accuracy_matrix = []
    question_list = []
    
    for q in questions:
        q_accuracies = []
        for model in models:
            model_q_data = df[(df['model'] == model) & (df['question'] == q)]
            if not model_q_data.empty:
                q_accuracies.append(model_q_data['correct'].mean())
            else:
                q_accuracies.append(np.nan)
        
        # Only include questions with data for all models
        if not any(np.isnan(q_accuracies)):
            accuracy_matrix.append(q_accuracies)
            question_list.append(q)
    
    if len(accuracy_matrix) < 2:
        print("Not enough complete data for clustering")
        return
    
    X = np.array(accuracy_matrix)
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # K-means clustering
    n_clusters = min(4, len(X))
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    clusters = kmeans.fit_predict(X_scaled)
    
    # PCA for visualization
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    # Plot clusters
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', alpha=0.6)
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
    plt.title('Question Clusters Based on Model Performance')
    plt.colorbar(scatter, label='Cluster')
    
    # Annotate cluster centers
    centers_pca = pca.transform(scaler.transform(kmeans.cluster_centers_))
    plt.scatter(centers_pca[:, 0], centers_pca[:, 1], c='red', marker='x', s=200, linewidths=3)
    
    plt.show()
    
    # Analyze cluster characteristics
    print("\n📊 Cluster Characteristics:")
    for i in range(n_clusters):
        cluster_mask = clusters == i
        cluster_questions = [question_list[j] for j, mask in enumerate(cluster_mask) if mask]
        cluster_accuracies = X[cluster_mask]
        
        print(f"\n  Cluster {i} ({sum(cluster_mask)} questions):")
        print(f"    Mean accuracy: {np.mean(cluster_accuracies):.3f}")
        print(f"    Std accuracy: {np.std(cluster_accuracies):.3f}")
        
        # Find characteristic questions
        if len(cluster_questions) > 0:
            sample_q = cluster_questions[0][:50] + "..." if len(cluster_questions[0]) > 50 else cluster_questions[0]
            print(f"    Sample question: {sample_q}")
    
    return clusters, X_pca

clusters, pca_results = perform_clustering_analysis(df)

## 12. Bayesian Analysis

In [ ]:

if BAYESIAN_AVAILABLE:
    def bayesian_analysis(df):
        """
        Bayesian analysis of model performance
        """
        print("\n" + "="*60)
        print("BAYESIAN ANALYSIS")
        print("="*60)
        
        models = df['model'].unique()
        
        # Prepare data
        model_accuracies = {}
        for model in models:
            model_data = df[df['model'] == model]
            successes = model_data['correct'].sum()
            trials = len(model_data)
            model_accuracies[model] = (successes, trials)
        
        # Bayesian model
        with pm.Model() as accuracy_model:
            # Prior for accuracy (weakly informative)
            theta = {}
            for model in models:
                theta[model] = pm.Beta(f'theta_{model}', alpha=2, beta=2)
            
            # Likelihood
            for model in models:
                successes, trials = model_accuracies[model]
                pm.Binomial(f'obs_{model}', n=trials, p=theta[model], observed=successes)
            
            # Sample
            trace = pm.sample(2000, tune=1000, return_inferencedata=True)
        
        # Plot posterior distributions
        fig, axes = plt.subplots(len(models), 2, figsize=(12, 3*len(models)))
        if len(models) == 1:
            axes = axes.reshape(1, -1)
        
        for i, model in enumerate(models):
            # Posterior distribution
            ax = axes[i, 0]
            posterior = trace.posterior[f'theta_{model}'].values.flatten()
            ax.hist(posterior, bins=50, density=True, alpha=0.7, edgecolor='black')
            ax.axvline(np.mean(posterior), color='red', linestyle='--', label=f'Mean: {np.mean(posterior):.3f}')
            ax.set_xlabel('Accuracy')
            ax.set_ylabel('Density')
            ax.set_title(f'{model} Posterior Distribution')
            ax.legend()
            
            # Credible intervals
            ax = axes[i, 1]
            ci_95 = np.percentile(posterior, [2.5, 97.5])
            ci_89 = np.percentile(posterior, [5.5, 94.5])
            
            ax.barh(['95% CI', '89% CI'], [ci_95[1]-ci_95[0], ci_89[1]-ci_89[0]], 
                   left=[ci_95[0], ci_89[0]], alpha=0.7)
            ax.set_xlabel('Accuracy')
            ax.set_title(f'{model} Credible Intervals')
            
        plt.tight_layout()
        plt.show()
        
        # Compare models
        print("\n📊 Pairwise Comparisons (Probability A > B):")
        for i, model1 in enumerate(models):
            for model2 in models[i+1:]:
                post1 = trace.posterior[f'theta_{model1}'].values.flatten()
                post2 = trace.posterior[f'theta_{model2}'].values.flatten()
                prob_better = np.mean(post1 > post2)
                
                print(f"  P({model1} > {model2}) = {prob_better:.3f}")
                
                if prob_better > 0.95:
                    print(f"    → Strong evidence {model1} is better")
                elif prob_better > 0.75:
                    print(f"    → Moderate evidence {model1} is better")
                elif prob_better < 0.25:
                    print(f"    → Moderate evidence {model2} is better")
                elif prob_better < 0.05:
                    print(f"    → Strong evidence {model2} is better")
                else:
                    print(f"    → No clear winner")
        
        return trace
    
    bayesian_results = bayesian_analysis(df)
else:
    print("\n⚠️ Bayesian analysis skipped (PyMC3 not installed)")

## 13. Time Complexity Analysis for Query/Response

In [ ]:

def analyze_response_time_patterns(df):
    """
    Analyze response time and length patterns
    """
    print("\n" + "="*60)
    print("RESPONSE COMPLEXITY ANALYSIS")
    print("="*60)
    
    # Response length by correctness
    for model in df['model'].unique():
        model_data = df[df['model'] == model]
        
        correct_lengths = model_data[model_data['correct'] == True]['response_length'].values
        incorrect_lengths = model_data[model_data['correct'] == False]['response_length'].values
        
        if len(correct_lengths) > 0 and len(incorrect_lengths) > 0:
            print(f"\n📝 {model}:")
            
            # Mann-Whitney U test (non-parametric)
            u_stat, p_val = mannwhitneyu(correct_lengths, incorrect_lengths)
            
            print(f"  Correct answers: {np.mean(correct_lengths):.0f} chars (median: {np.median(correct_lengths):.0f})")
            print(f"  Incorrect answers: {np.mean(incorrect_lengths):.0f} chars (median: {np.median(incorrect_lengths):.0f})")
            print(f"  Mann-Whitney U test: U={u_stat:.1f}, p={p_val:.4f}")
            
            if p_val < 0.05:
                if np.median(correct_lengths) > np.median(incorrect_lengths):
                    print(f"  → Correct answers are significantly longer")
                else:
                    print(f"  → Incorrect answers are significantly longer")
            
            # Correlation between length and accuracy
            correlation, p_corr = pearsonr(model_data['response_length'], model_data['correct'])
            print(f"  Length-accuracy correlation: r={correlation:.3f}, p={p_corr:.4f}")

analyze_response_time_patterns(df)

## 14. Multiple Comparisons Correction


In [ ]:

def apply_multiple_comparisons_correction(p_values):
    """
    Apply various multiple comparisons corrections
    """
    from statsmodels.stats.multitest import multipletests
    
    print("\n" + "="*60)
    print("MULTIPLE COMPARISONS CORRECTION")
    print("="*60)
    
    if not p_values:
        print("No p-values to correct")
        return
    
    # Extract p-values
    test_names = list(p_values.keys())
    p_vals = list(p_values.values())
    
    # Apply different corrections
    corrections = {
        'Bonferroni': 'bonferroni',
        'Holm': 'holm',
        'Benjamini-Hochberg (FDR)': 'fdr_bh',
        'Benjamini-Yekutieli': 'fdr_by'
    }
    
    results = {}
    
    for name, method in corrections.items():
        reject, p_corrected, _, _ = multipletests(p_vals, alpha=0.05, method=method)
        results[name] = list(zip(test_names, p_vals, p_corrected, reject))
        
        print(f"\n📊 {name} Correction:")
        print(f"  {'Test':<30} {'Original p':<12} {'Corrected p':<12} {'Significant'}")
        print("  " + "-"*70)
        
        for test, p_orig, p_corr, sig in results[name]:
            sig_marker = "***" if p_corr < 0.001 else "**" if p_corr < 0.01 else "*" if p_corr < 0.05 else ""
            print(f"  {test:<30} {p_orig:<12.4f} {p_corr:<12.4f} {sig_marker}")
    
    return results

# Example: collect p-values from previous tests
example_p_values = {
    'Position_Bias_Model1': 0.023,
    'MCQ_vs_OSQ_Model1': 0.045,
    'Position_A_vs_B': 0.012,
    'Position_A_vs_C': 0.067,
    'Position_A_vs_D': 0.089,
    'Model1_vs_Model2': 0.003,
    'Response_Length_Effect': 0.034
}

corrected_results = apply_multiple_comparisons_correction(example_p_values)